In [1]:
import json
import pandas as pd
import duckdb
import numpy as np
import glob
from pathlib import Path

# Lendo dados

In [2]:
caminhos_arquivos = glob.glob('dados_brutos/professores/ufrj/*.json')

In [3]:
# 1. LISTAS PARA ACUMULAR OS DADOS
# ---------------------------------------------------------
# Dados Gerais
lista_pessoas = []
lista_bancas = []
lista_eventos = []
lista_orientacoes = []
lista_premios = []
lista_projetos = []

# Produção Bibliográfica
lista_bib_artigos = []
lista_bib_livros = []
lista_bib_capitulos = []
lista_bib_trabalhos_congresso = []
lista_bib_resumos_expandidos = []
lista_bib_resumos_congresso = []
lista_bib_artigos_aceitos = []
lista_bib_apresentacoes = []
lista_bib_textos_jornais = []
lista_bib_outras = []

# Produção Técnica
lista_tec_softwares_patente = []
lista_tec_softwares_sem_patente = []
lista_tec_produtos = []
lista_tec_processos = []
lista_tec_trabalhos = []
lista_tec_outras = []
lista_tec_entrevistas = []

# Patentes e Registros
lista_pat_patentes = []
lista_pat_programas = []
lista_pat_desenhos = []

print("Iniciando o processamento dos arquivos JSON...")
#caminhos_arquivos = glob.glob('dados_brutos/*.json')

if not caminhos_arquivos:
    print("ERRO: Nenhum arquivo JSON encontrado na pasta 'dados_brutos/'.")
    exit()

# ---------------------------------------------------------
# 2. EXTRAÇÃO E ACHATAMENTO (FLATTEN)
# ---------------------------------------------------------
for arquivo in caminhos_arquivos:
    with open(arquivo, 'r', encoding='utf-8') as f:
        dados = json.load(f)
        
        id_lattes = dados.get('informacoes_pessoais', {}).get('id_lattes')
        if not id_lattes:
            continue
            
        # --- PESSOAS ---
        df_pessoa = pd.json_normalize(dados['informacoes_pessoais'])
        lista_pessoas.append(df_pessoa)
        
        # --- BANCAS ---
        if 'bancas' in dados:
            for categoria, itens in dados['bancas'].items():
                if itens:
                    df_temp = pd.DataFrame(itens)
                    df_temp['id_lattes'] = id_lattes
                    df_temp['categoria_banca'] = categoria 
                    if 'membros_banca' in df_temp.columns:
                        df_temp['membros_banca'] = df_temp['membros_banca'].astype(str)
                    lista_bancas.append(df_temp)

        # --- EVENTOS ---
        if 'eventos' in dados:
            for categoria, itens in dados['eventos'].items():
                if itens:
                    df_temp = pd.DataFrame(itens)
                    df_temp['id_lattes'] = id_lattes
                    df_temp['categoria_evento'] = categoria
                    lista_eventos.append(df_temp)

        # --- ORIENTAÇÕES ---
        if 'orientacoes' in dados:
            for status, dicionario_niveis in dados['orientacoes'].items():
                for nivel, itens in dicionario_niveis.items():
                    if itens:
                        df_temp = pd.DataFrame(itens)
                        df_temp['id_lattes'] = id_lattes
                        df_temp['status'] = status
                        df_temp['nivel'] = nivel   
                        lista_orientacoes.append(df_temp)

        # --- PRÊMIOS ---
        if 'premios_titulos' in dados and dados['premios_titulos']:
            df_temp = pd.DataFrame(dados['premios_titulos'])
            df_temp['id_lattes'] = id_lattes
            lista_premios.append(df_temp)

        # --- PROJETOS ---
        if 'projetos_pesquisa' in dados and dados['projetos_pesquisa']:
            df_temp = pd.DataFrame(dados['projetos_pesquisa'])
            df_temp['id_lattes'] = id_lattes
            for col in ['descricao', 'integrantes', 'financiadores']:
                if col in df_temp.columns:
                    df_temp[col] = df_temp[col].astype(str)
            lista_projetos.append(df_temp)

        # --- PRODUÇÃO BIBLIOGRÁFICA ---
        prod_bib = dados.get('producao_bibliografica', {})
        def add_to_list(chave, lista_destino):
            itens = prod_bib.get(chave, [])
            if itens:
                df_temp = pd.DataFrame(itens)
                df_temp['id_lattes'] = id_lattes
                lista_destino.append(df_temp)

        add_to_list('artigos_periodicos', lista_bib_artigos)
        add_to_list('livros_publicados', lista_bib_livros)
        add_to_list('capitulos_livros', lista_bib_capitulos)
        add_to_list('trabalhos_completos_congressos', lista_bib_trabalhos_congresso)
        add_to_list('resumos_expandidos', lista_bib_resumos_expandidos)
        add_to_list('resumos_congressos', lista_bib_resumos_congresso)
        add_to_list('artigos_aceitos', lista_bib_artigos_aceitos)
        add_to_list('apresentacoes_trabalhos', lista_bib_apresentacoes)
        add_to_list('textos_jornais', lista_bib_textos_jornais)
        add_to_list('outras_producoes', lista_bib_outras)

        # --- PRODUÇÃO TÉCNICA ---
        prod_tec = dados.get('producao_tecnica', {})
        def add_to_list_tec(chave, lista_destino):
            itens = prod_tec.get(chave, [])
            if itens:
                df_temp = pd.DataFrame(itens)
                df_temp['id_lattes'] = id_lattes
                lista_destino.append(df_temp)

        add_to_list_tec('softwares_com_patente', lista_tec_softwares_patente)
        add_to_list_tec('softwares_sem_patente', lista_tec_softwares_sem_patente)
        add_to_list_tec('produtos_tecnologicos', lista_tec_produtos)
        add_to_list_tec('processos_tecnicas', lista_tec_processos)
        add_to_list_tec('trabalhos_tecnicos', lista_tec_trabalhos)
        add_to_list_tec('outras_producoes_tecnicas', lista_tec_outras)
        add_to_list_tec('entrevistas', lista_tec_entrevistas)

        # --- PATENTES ---
        patentes = dados.get('patentes_registros', {})
        def add_to_list_pat(chave, lista_destino):
            itens = patentes.get(chave, [])
            if itens:
                df_temp = pd.DataFrame(itens)
                df_temp['id_lattes'] = id_lattes
                lista_destino.append(df_temp)

        add_to_list_pat('patentes', lista_pat_patentes)
        add_to_list_pat('programas_computador', lista_pat_programas)
        add_to_list_pat('desenhos_industriais', lista_pat_desenhos)


# ---------------------------------------------------------
# 3. CONSOLIDAÇÃO EM DATAFRAMES EXPLÍCITOS
# ---------------------------------------------------------
print("Consolidando DataFrames...")

def consolidar(lista):
    return pd.concat(lista, ignore_index=True) if lista else pd.DataFrame()

# DataFrames Gerais
df_pessoas = consolidar(lista_pessoas)
df_bancas = consolidar(lista_bancas)
df_eventos = consolidar(lista_eventos)
df_orientacoes = consolidar(lista_orientacoes)
df_premios = consolidar(lista_premios)
df_projetos = consolidar(lista_projetos)

# DataFrames Produção Bibliográfica
df_bib_artigos = consolidar(lista_bib_artigos)
df_bib_livros = consolidar(lista_bib_livros)
df_bib_capitulos = consolidar(lista_bib_capitulos)
df_bib_trab_congresso = consolidar(lista_bib_trabalhos_congresso)
df_bib_resumos_exp = consolidar(lista_bib_resumos_expandidos)
df_bib_resumos_cong = consolidar(lista_bib_resumos_congresso)
df_bib_art_aceitos = consolidar(lista_bib_artigos_aceitos)
df_bib_apresentacoes = consolidar(lista_bib_apresentacoes)
df_bib_textos_jornais = consolidar(lista_bib_textos_jornais)
df_bib_outras = consolidar(lista_bib_outras)

# DataFrames Produção Técnica
df_tec_soft_patente = consolidar(lista_tec_softwares_patente)
df_tec_soft_sem_patente = consolidar(lista_tec_softwares_sem_patente)
df_tec_produtos = consolidar(lista_tec_produtos)
df_tec_processos = consolidar(lista_tec_processos)
df_tec_trabalhos = consolidar(lista_tec_trabalhos)
df_tec_outras = consolidar(lista_tec_outras)
df_tec_entrevistas = consolidar(lista_tec_entrevistas)

# DataFrames Patentes
df_pat_patentes = consolidar(lista_pat_patentes)
df_pat_programas = consolidar(lista_pat_programas)
df_pat_desenhos = consolidar(lista_pat_desenhos)

Iniciando o processamento dos arquivos JSON...
Consolidando DataFrames...


# Tratando dados

## Informações pessoais

In [4]:
df_pessoas.info()

<class 'pandas.DataFrame'>
RangeIndex: 40 entries, 0 to 39
Data columns (total 11 columns):
 #   Column                 Non-Null Count  Dtype
---  ------                 --------------  -----
 0   id_lattes              40 non-null     str  
 1   nome_completo          40 non-null     str  
 2   nome_citacoes          40 non-null     str  
 3   sexo                   40 non-null     str  
 4   rotulo                 40 non-null     str  
 5   periodo                40 non-null     str  
 6   bolsa_produtividade    40 non-null     str  
 7   endereco_profissional  40 non-null     str  
 8   atualizacao_cv         40 non-null     str  
 9   url                    40 non-null     str  
 10  texto_resumo           40 non-null     str  
dtypes: str(11)
memory usage: 80.0 KB


In [5]:
# 1. Substituir strings vazias e espaços em branco por NaN
# Usa expressão regular para pegar "" ou "   "
df_pessoas.replace(r'^\s*$', np.nan, regex=True, inplace=True)

,id_lattes,nome_completo,nome_citacoes,sexo,rotulo,periodo,bolsa_produtividade,endereco_profissional,atualizacao_cv,url,texto_resumo
0,0211300683784278,Márcia Rosana Cerioli,"CERIOLI, M. R.;Cerioli, M;Cerioli, Marcia R.;C...",Masculino,* Sem rótulo,NaN,NaN,"Universidade Federal do Rio de Janeiro, Instit...",30/03/2026,http://lattes.cnpq.br/0211300683784278,Possui graduação em Matemática pela Universida...
1,1511298432327033,Daniel Serrão Schneider,"SCHNEIDER, D. S.;SCHNEIDER, DANIEL;SCHNEIDER, ...",Masculino,* Sem rótulo,NaN,NaN,"Universidade Federal do Rio de Janeiro, Instit...",20/01/2026,http://lattes.cnpq.br/1511298432327033,Doutor em Engenharia de Sistemas e Computação ...
2,9770406908381251,Claudio Luis de Amorim,"AMORIM, C. L.;AMORIM, CLAUDIO LUIS DE;AMORIM, ...",Masculino,* Sem rótulo,NaN,NaN,"Universidade Federal do Rio de Janeiro, Instit...",05/12/2024,http://lattes.cnpq.br/9770406908381251,Professor Titular do Programa de Engenharia de...
3,8154171198308578,Fábio Happ Botler,"BOTLER, F. H.;BOTLER, F.;BOTLER, FÁBIO;BOTLER,...",Masculino,* Sem rótulo,NaN,Nível 2,"Universidade de São Paulo, Instituto de Matemá...",12/03/2026,http://lattes.cnpq.br/8154171198308578,Possui graduação e mestrado em Matemática pela...
4,6243465206463403,Claudio Miceli de Farias,"FARIAS, C. M.;Farias, Claudio M.;Miceli, C.;MI...",Masculino,* Sem rótulo,NaN,NaN,"Universidade Federal do Rio de Janeiro, Instit...",30/07/2025,http://lattes.cnpq.br/6243465206463403,O professor Claudio Miceli de Farias fez gradu...
5,0523104569378276,Marcia Helena Costa Fampa,"FAMPA, M. H. C.;FAMPA, M.;FAMPA, MARCIA H.C.;F...",Masculino,* Sem rótulo,NaN,Nível 2,"Universidade Federal do Rio de Janeiro, Progra...",07/04/2026,http://lattes.cnpq.br/0523104569378276,Marcia é Professora da Universidade Federal do...
6,2002515486942024,Jayme Luiz Szwarcfiter,"SZWARCFITER, J. L.;Szwarcfiter, Jayme L.;Jayme...",Masculino,* Sem rótulo,NaN,Nível 1A (***,"Universidade Federal do Rio de Janeiro, COPPE ...",05/09/2025,http://lattes.cnpq.br/2002515486942024,Possui graduação em Engenharia Eletrônica pela...
7,9358511568098561,Edmundo Albuquerque de Souza e Silva,"de Souza e Silva, E.;de Souza e Silva, Edmundo...",Masculino,* Sem rótulo,NaN,Nível SR,"Universidade Federal do Rio de Janeiro, Instit...",15/01/2025,http://lattes.cnpq.br/9358511568098561,Edmundo de Souza e Silva é Engenheiro Elétrico...
8,5815607228657970,Henrique Luiz Cukierman,"CUKIERMAN, H. L.;CUKIERMAN, HENRIQUE LUIZ;CUKI...",Masculino,* Sem rótulo,NaN,NaN,"Universidade Federal do Rio de Janeiro, Instit...",08/11/2025,http://lattes.cnpq.br/5815607228657970,Possui graduação em Engenharia de Sistemas pel...
9,2704717555047499,Priscila Machado Vieira Lima,"LIMA, P. M. V.;LIMA, PRISCILA M. V.;LIMA, PRIS...",Masculino,* Sem rótulo,NaN,NaN,"Universidade Federal do Rio de Janeiro, Núcleo...",08/08/2025,http://lattes.cnpq.br/2704717555047499,Possui graduação em Informática pela Universid...


In [6]:
# 2. Tratamento da Data de Atualização
# Converte a string '15/10/2025' para um tipo datetime
if 'atualizacao_cv' in df_pessoas.columns:
    df_pessoas['atualizacao_cv'] = pd.to_datetime(
        df_pessoas['atualizacao_cv'], 
        format='%d/%m/%Y', 
        errors='coerce' # Se tiver uma data bizarra (ex: 99/99/9999), vira nulo em vez de quebrar o script
    )

In [7]:
# 3. Limpeza do campo Rótulo
if 'rotulo' in df_pessoas.columns:
    # Remove o asterisco e espaços em branco nas pontas
    df_pessoas['rotulo'] = df_pessoas['rotulo'].str.replace('*', '', regex=False).str.strip()
    # Se o rótulo ficou "Sem rótulo", transforma em nulo verdadeiro
    df_pessoas['rotulo'] = df_pessoas['rotulo'].replace('Sem rótulo', np.nan)

In [8]:
# 5. Garantia de Tipagem da Chave Primária e Textos Longos
df_pessoas['id_lattes'] = df_pessoas['id_lattes'].astype(str)

In [9]:
if 'texto_resumo' in df_pessoas.columns:
    # Tira quebras de linha e espaços duplos extras no começo e no fim do resumo
    df_pessoas['texto_resumo'] = df_pessoas['texto_resumo'].str.strip()

In [10]:
if 'texto_resumo' in df_pessoas.columns:
    # Tira quebras de linha e espaços duplos extras no começo e no fim do resumo
    df_pessoas['texto_resumo'] = df_pessoas['texto_resumo'].str.strip()

In [11]:
df_pessoas.head()

,id_lattes,nome_completo,nome_citacoes,sexo,rotulo,periodo,bolsa_produtividade,endereco_profissional,atualizacao_cv,url,texto_resumo
0,0211300683784278,Márcia Rosana Cerioli,"CERIOLI, M. R.;Cerioli, M;Cerioli, Marcia R.;C...",Masculino,NaN,NaN,NaN,"Universidade Federal do Rio de Janeiro, Instit...",2026-03-30,http://lattes.cnpq.br/0211300683784278,Possui graduação em Matemática pela Universida...
1,1511298432327033,Daniel Serrão Schneider,"SCHNEIDER, D. S.;SCHNEIDER, DANIEL;SCHNEIDER, ...",Masculino,NaN,NaN,NaN,"Universidade Federal do Rio de Janeiro, Instit...",2026-01-20,http://lattes.cnpq.br/1511298432327033,Doutor em Engenharia de Sistemas e Computação ...
2,9770406908381251,Claudio Luis de Amorim,"AMORIM, C. L.;AMORIM, CLAUDIO LUIS DE;AMORIM, ...",Masculino,NaN,NaN,NaN,"Universidade Federal do Rio de Janeiro, Instit...",2024-12-05,http://lattes.cnpq.br/9770406908381251,Professor Titular do Programa de Engenharia de...
3,8154171198308578,Fábio Happ Botler,"BOTLER, F. H.;BOTLER, F.;BOTLER, FÁBIO;BOTLER,...",Masculino,NaN,NaN,Nível 2,"Universidade de São Paulo, Instituto de Matemá...",2026-03-12,http://lattes.cnpq.br/8154171198308578,Possui graduação e mestrado em Matemática pela...
4,6243465206463403,Claudio Miceli de Farias,"FARIAS, C. M.;Farias, Claudio M.;Miceli, C.;MI...",Masculino,NaN,NaN,NaN,"Universidade Federal do Rio de Janeiro, Instit...",2025-07-30,http://lattes.cnpq.br/6243465206463403,O professor Claudio Miceli de Farias fez gradu...


## Tratando orientações

In [12]:
df_orientacoes.head()

,titulo,ano_inicio,orientando,tipo_trabalho,instituicao,curso,id_lattes,status,nivel,ano_conclusao
0,Propriedades estruturais de grafos cordais e c...,2025,Rodrigo Fernandes Souto,Tese,Universidade Federal do Rio de Janeiro,,0211300683784278,em_andamento,doutorado,NaN
1,More on set graphs,2022,Bruno Bandeira Monteiro,Tese,Universidade Federal do Rio de Janeiro,,0211300683784278,em_andamento,doutorado,NaN
2,Colorações Seletivas em Grafos,2026,Rafael Paladini Meirelles,Dissertação,Universidade Federal do Rio de Janeiro,,0211300683784278,em_andamento,mestrado,NaN
3,Algoritmos de caminho em grafos,2025,Eduardo Naslausky,Dissertação,Universidade Federal do Rio de Janeiro,,0211300683784278,em_andamento,mestrado,NaN
4,Algoritmo de Caminho Mínimo: uma atividade de ...,2023,Caio de Campos,Trabalho de Conclusão de Curso,Universidade Federal do Rio de Janeiro,,0211300683784278,em_andamento,tcc,NaN


In [13]:
print(df_orientacoes['tipo_trabalho'].value_counts())
print(df_orientacoes['status'].value_counts())
print(df_orientacoes['nivel'].value_counts())
print(df_orientacoes['curso'].value_counts())

tipo_trabalho
Dissertação                       1274
Trabalho de Conclusão de Curso     707
Tese                               643
                                   207
Monografia                          32
Trabalho                            30
Iniciação científica                14
Name: count, dtype: int64
status
concluidas      2671
em_andamento     236
Name: count, dtype: int64
nivel
mestrado                1301
doutorado                651
tcc                      451
iniciacao_cientifica     346
pos_doutorado             81
especializacao            44
outros                    33
Name: count, dtype: int64
curso
    2907
Name: count, dtype: int64


In [14]:
df_orientacoes = df_orientacoes.rename(columns={'titulo': 'titulo_trabalho'})

In [15]:
import pandas as pd

print("Iniciando o tratamento da tabela de orientações...")

# 1. Tratamento de Strings: Remove espaços duplos e quebras de linha escondidas
colunas_texto = ['titulo_trabalho', 'orientando', 'tipo_trabalho', 'instituicao', 'curso']
for col in colunas_texto:
    df_orientacoes[col] = df_orientacoes[col].astype(str).str.strip()
    # Se, após limpar os espaços, o campo ficar vazio ou 'nan', preenche com 'Não informado'
    df_orientacoes[col] = df_orientacoes[col].replace({'': 'Não informado', 'nan': 'Não informado', 'None': 'Não informado'})

# 2. Conversão segura de anos (de float para Int64 com suporte a Nulo)
df_orientacoes['ano_conclusao'] = df_orientacoes['ano_conclusao'].astype('Int64')

# 3. Melhoria estética na coluna 'Nível' para os gráficos do Streamlit
mapeamento_nivel = {
    'mestrado': 'Mestrado',
    'doutorado': 'Doutorado',
    'tcc': 'TCC',
    'iniciacao_cientifica': 'Iniciação Científica',
    'pos_doutorado': 'Pós-Doutorado',
    'especializacao': 'Especialização',
    'outros': 'Outros'
}
df_orientacoes['nivel'] = df_orientacoes['nivel'].map(mapeamento_nivel).fillna(df_orientacoes['nivel'])

# 4. Melhoria estética na coluna 'Status' para os gráficos
mapeamento_status = {
    'concluidas': 'Concluída',
    'em_andamento': 'Em Andamento'
}
df_orientacoes['status'] = df_orientacoes['status'].map(mapeamento_status).fillna(df_orientacoes['status'])

print("\nTratamento concluído com sucesso! Verifique a nova estrutura:")
df_orientacoes.info()

print("\nAmostra dos dados tratados:")
display(df_orientacoes[['orientando', 'nivel', 'status', 'ano_conclusao']].head())

Iniciando o tratamento da tabela de orientações...

Tratamento concluído com sucesso! Verifique a nova estrutura:
<class 'pandas.DataFrame'>
RangeIndex: 2907 entries, 0 to 2906
Data columns (total 10 columns):
 #   Column           Non-Null Count  Dtype
---  ------           --------------  -----
 0   titulo_trabalho  2907 non-null   str  
 1   ano_inicio       2907 non-null   int64
 2   orientando       2907 non-null   str  
 3   tipo_trabalho    2907 non-null   str  
 4   instituicao      2907 non-null   str  
 5   curso            2907 non-null   str  
 6   id_lattes        2907 non-null   str  
 7   status           2907 non-null   str  
 8   nivel            2907 non-null   str  
 9   ano_conclusao    2671 non-null   Int64
dtypes: Int64(1), int64(1), str(8)
memory usage: 817.5 KB

Amostra dos dados tratados:


,orientando,nivel,status,ano_conclusao
0,Rodrigo Fernandes Souto,Doutorado,Em Andamento,<NA>
1,Bruno Bandeira Monteiro,Doutorado,Em Andamento,<NA>
2,Rafael Paladini Meirelles,Mestrado,Em Andamento,<NA>
3,Eduardo Naslausky,Mestrado,Em Andamento,<NA>
4,Caio de Campos,TCC,Em Andamento,<NA>


## Informações acerca de periódicos publicados

In [16]:
df_bib_artigos.head()

,titulo,ano,autores,revista,volume,numero,paginas,issn,doi,qualis,id_lattes
0,On the (In)Dependence of the Peano Axioms for ...,2021,"CERIOLI, MÁRCIA R.; NOBREGA, HUGO ; SILVEIRA, ...",History and Philosophy of Logic,?,,1-19,1464-5149,http://dx.doi.org/10.1080/01445340.2021.1971005,,0211300683784278
1,Short proofs on the structure of general parti...,2021,"CERIOLI, MÁRCIA R.; MARTINS, TAÍSA",DISCRETE APPLIED MATHEMATICS,303,,8-13,0166-218X,http://dx.doi.org/10.1016/j.dam.2020.09.007,,0211300683784278
2,Transversals of longest paths,2020,"CERIOLI, MÁRCIA R.; FERNANDES, CRISTINA G. ; G...",DISCRETE MATHEMATICS,343,,111717,0012-365X,http://dx.doi.org/10.1016/j.disc.2019.111717,,0211300683784278
3,Intersection of longest paths in graph classes,2020,"CERIOLI, MÁRCIA R.; LIMA, PALOMA T.",DISCRETE APPLIED MATHEMATICS,281,,96-105,0166-218X,http://dx.doi.org/10.1016/j.dam.2019.03.022,,0211300683784278
4,On Edge-magic Labelings of Forests,2019,"CERIOLI, M. R.; FERNANDES, C. G. ; LEE, O. ; L...",ELECTRONIC NOTES IN THEORETICAL COMPUTER SCIENCE,346,,299-307,1571-0661,http://dx.doi.org/10.1016/j.entcs.2019.08.027,,0211300683784278


In [17]:
print("Aplicando tratamentos na tabela 'bib_artigos'...")

if not df_bib_artigos.empty:
    
    # 1. Transformar strings vazias ou só com espaços em nulos reais (NaN)
    df_bib_artigos.replace(r'^\s*$', np.nan, regex=True, inplace=True)
    
    # 2. Tratamento da Revista (Periódico)
    if 'revista' in df_bib_artigos.columns:
        # Força maiúsculo e remove espaços extras no início e no fim
        df_bib_artigos['revista'] = df_bib_artigos['revista'].str.upper().str.strip()

    # 4. Tratamento do Ano (Garantir que seja número inteiro)
    if 'ano' in df_bib_artigos.columns:
        # errors='coerce' transforma erros (ex: "Sem ano") em NaN
        # Int64 é o tipo inteiro do Pandas que aceita valores nulos
        df_bib_artigos['ano'] = pd.to_numeric(df_bib_artigos['ano'], errors='coerce').astype('Int64')

    # 5. Tratamento de Título, DOI e ISSN (Apenas remover espaços ocultos)
    colunas_texto = ['titulo', 'doi', 'issn', 'volume', 'numero', 'paginas']
    for col in colunas_texto:
        if col in df_bib_artigos.columns:
            df_bib_artigos[col] = df_bib_artigos[col].str.strip()

    # 6. Garantir tipagem da chave primária
    df_bib_artigos['id_lattes'] = df_bib_artigos['id_lattes'].astype(str)

print("Tratamento da tabela 'bib_artigos' concluído!")
display(df_bib_artigos[['ano', 'revista', 'doi', 'issn']].head())

Aplicando tratamentos na tabela 'bib_artigos'...
Tratamento da tabela 'bib_artigos' concluído!


,ano,revista,doi,issn
0,2021,HISTORY AND PHILOSOPHY OF LOGIC,http://dx.doi.org/10.1080/01445340.2021.1971005,1464-5149
1,2021,DISCRETE APPLIED MATHEMATICS,http://dx.doi.org/10.1016/j.dam.2020.09.007,0166-218X
2,2020,DISCRETE MATHEMATICS,http://dx.doi.org/10.1016/j.disc.2019.111717,0012-365X
3,2020,DISCRETE APPLIED MATHEMATICS,http://dx.doi.org/10.1016/j.dam.2019.03.022,0166-218X
4,2019,ELECTRONIC NOTES IN THEORETICAL COMPUTER SCIENCE,http://dx.doi.org/10.1016/j.entcs.2019.08.027,1571-0661


In [18]:
df_bib_artigos.info()

<class 'pandas.DataFrame'>
RangeIndex: 2018 entries, 0 to 2017
Data columns (total 11 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   titulo     2018 non-null   str  
 1   ano        2018 non-null   Int64
 2   autores    2018 non-null   str  
 3   revista    2018 non-null   str  
 4   volume     1981 non-null   str  
 5   numero     220 non-null    str  
 6   paginas    2001 non-null   str  
 7   issn       1977 non-null   str  
 8   doi        1531 non-null   str  
 9   qualis     0 non-null      str  
 10  id_lattes  2018 non-null   str  
dtypes: Int64(1), str(10)
memory usage: 674.2 KB


## Tratando informações de eventos

In [19]:
df_bib_trab_congresso.head()

,titulo,ano,doi,autores,evento,cidade,paginas,isbn,id_lattes
0,Another Calculational Proof of Cantor's Theorem,2022,http://dx.doi.org/10.5753/wbl.2022.223244,"CERIOLI, MÁRCIA R.; FREITAS, RENATA DE ; VIANA...",Workshop Brasileiro de Lógica,,9,,0211300683784278
1,Presenting Basic Graph Logic,2021,http://dx.doi.org/10.1007/978-3-030-86062-2,"CERIOLI, M. R.; SUGUITANI, L. ; VIANA, PETRUCIO",Diagrams,,132-148,,0211300683784278
2,Transversals of Longest Paths,2017,,"CERIOLI, M. R.; FERNANDES, C. G. ; GOMES, R. ;...",Latin and American Algorithms,,,,0211300683784278
3,On the (in)dependence of the Dedekind-Peano ax...,2017,http://dx.doi.org/10.5540/03.2017.005.01.0239,"CERIOLI, MA'RCIA; NOBREGA, HUGO ; SILVEIRA, GU...",CNMAC 2016 XXXVI Congresso Nacional de Matemát...,,,,0211300683784278
4,"L(2, 1)-coloração de k-árvores e grafos com tr...",2015,http://dx.doi.org/10.5540/03.2015.003.01.0241,"BARROS, GABRIEL F. ; POSNER, DANIEL F. D. ; CE...",XXXV CNMAC Congresso Nacional de Matemática Ap...,,,,0211300683784278


In [20]:
df_bib_trab_congresso.info()

<class 'pandas.DataFrame'>
RangeIndex: 3670 entries, 0 to 3669
Data columns (total 9 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   titulo     3670 non-null   str  
 1   ano        3670 non-null   int64
 2   doi        3670 non-null   str  
 3   autores    3670 non-null   str  
 4   evento     3670 non-null   str  
 5   cidade     3670 non-null   str  
 6   paginas    3670 non-null   str  
 7   isbn       3670 non-null   str  
 8   id_lattes  3670 non-null   str  
dtypes: int64(1), str(8)
memory usage: 1.1 MB


In [21]:
import numpy as np
import pandas as pd

print("Aplicando tratamentos na tabela 'df_bib_trab_congresso'...")

if not df_bib_trab_congresso.empty:
    
    # 1. Transformar strings vazias ou só com espaços em nulos reais (NaN)
    df_bib_trab_congresso.replace(r'^\s*$', np.nan, regex=True, inplace=True)
    
    # 2. Tratamento da coluna 'evento'
    if 'evento' in df_bib_trab_congresso.columns:
        df_bib_trab_congresso['evento'] = df_bib_trab_congresso['evento'].str.upper().str.strip()

    # 3. Tratamento da coluna 'ano'
    if 'ano' in df_bib_trab_congresso.columns:
        df_bib_trab_congresso['ano'] = pd.to_numeric(df_bib_trab_congresso['ano'], errors='coerce').astype('Int64')

    # 4. Tratamento de Título, DOI, ISBN e Paginas
    colunas_texto = ['titulo', 'doi', 'isbn', 'paginas']
    for col in colunas_texto:
        if col in df_bib_trab_congresso.columns:
            df_bib_trab_congresso[col] = df_bib_trab_congresso[col].str.strip()

    # 5. NOVO: Tratamento da coluna 'autores'
    if 'autores' in df_bib_trab_congresso.columns:
        # Remove espaços extras nas extremidades
        df_bib_trab_congresso['autores'] = df_bib_trab_congresso['autores'].str.strip()
        # Normaliza separadores: substitui ponto e vírgula por vírgula para manter padrão
        df_bib_trab_congresso['autores'] = df_bib_trab_congresso['autores'].str.replace(';', ',', regex=False)
        # Remove espaços duplos entre nomes/iniciais
        df_bib_trab_congresso['autores'] = df_bib_trab_congresso['autores'].str.replace(r'\s+', ' ', regex=True)
        # Opcional: Converter para maiúsculas para facilitar buscas
        df_bib_trab_congresso['autores'] = df_bib_trab_congresso['autores'].str.upper()

    # 6. Garantir tipagem da chave primária
    df_bib_trab_congresso['id_lattes'] = df_bib_trab_congresso['id_lattes'].astype(str)

print("Tratamento da tabela 'df_bib_trab_congresso' concluído!")
display(df_bib_trab_congresso[['ano', 'evento', 'autores']].head())

Aplicando tratamentos na tabela 'df_bib_trab_congresso'...
Tratamento da tabela 'df_bib_trab_congresso' concluído!


,ano,evento,autores
0,2022,WORKSHOP BRASILEIRO DE LÓGICA,"CERIOLI, MÁRCIA R., FREITAS, RENATA DE , VIANA..."
1,2021,DIAGRAMS,"CERIOLI, M. R., SUGUITANI, L. , VIANA, PETRUCIO"
2,2017,LATIN AND AMERICAN ALGORITHMS,"CERIOLI, M. R., FERNANDES, C. G. , GOMES, R. ,..."
3,2017,CNMAC 2016 XXXVI CONGRESSO NACIONAL DE MATEMÁT...,"CERIOLI, MA'RCIA, NOBREGA, HUGO , SILVEIRA, GU..."
4,2015,XXXV CNMAC CONGRESSO NACIONAL DE MATEMÁTICA AP...,"BARROS, GABRIEL F. , POSNER, DANIEL F. D. , CE..."


In [22]:
df_bib_trab_congresso.head(3)

,titulo,ano,doi,autores,evento,cidade,paginas,isbn,id_lattes
0,Another Calculational Proof of Cantor's Theorem,2022,http://dx.doi.org/10.5753/wbl.2022.223244,"CERIOLI, MÁRCIA R., FREITAS, RENATA DE , VIANA...",WORKSHOP BRASILEIRO DE LÓGICA,NaN,9,NaN,0211300683784278
1,Presenting Basic Graph Logic,2021,http://dx.doi.org/10.1007/978-3-030-86062-2,"CERIOLI, M. R., SUGUITANI, L. , VIANA, PETRUCIO",DIAGRAMS,NaN,132-148,NaN,0211300683784278
2,Transversals of Longest Paths,2017,NaN,"CERIOLI, M. R., FERNANDES, C. G. , GOMES, R. ,...",LATIN AND AMERICAN ALGORITHMS,NaN,NaN,NaN,0211300683784278


## Tratando informações de classificação

### Database de periódicos

In [23]:
import pandas as pd
import numpy as np

# ==========================================
# FUNÇÃO AUXILIAR DE LIMPEZA
# ==========================================
def formatar_issn(issn):
    # Converte para string, remove traços e tira espaços
    issn_str = str(issn).replace('-', '').strip()
    # Se for um valor nulo/vazio, retorna pd.NA para não virar "00000nan"
    if issn_str.lower() in ['nan', 'none', '', 'nat']:
        return pd.NA
    # Preenche com zeros à esquerda até completar 8 caracteres
    return issn_str.zfill(8)

print("Etapa 1: Carregando e preparando a base completa da Scopus...")
# Carrega o ficheiro original intacto da Scopus
df_scopus_raw = pd.read_excel('periodicos_percentil.xlsx')

# Limpeza de segurança padrão nas chaves de cruzamento 
df_scopus_raw['Title'] = df_scopus_raw['Title'].astype(str).str.upper().str.strip()

# Aplica a nova função de formatação de ISSN (8 dígitos)
df_scopus_raw['E-ISSN'] = df_scopus_raw['E-ISSN'].apply(formatar_issn)
df_scopus_raw['Print ISSN'] = df_scopus_raw['Print ISSN'].apply(formatar_issn)

# Mapeia TODOS os títulos/ISSNs que possuem alguma subárea de computação
mask_comput = df_scopus_raw['Scopus Sub-Subject Area'].str.contains('Comput', case=False, na=False)
titulos_computacao = set(df_scopus_raw[mask_comput]['Title'].unique())

# Unimos E-ISSN e Print ISSN no mesmo set para facilitar a checagem mais à frente
issns_computacao = set(df_scopus_raw[mask_comput]['E-ISSN'].dropna().unique()).union(
                   set(df_scopus_raw[mask_comput]['Print ISSN'].dropna().unique()))

# Ordena pelo Percentile (do maior para o menor) e remove duplicados de títulos.
df_scopus_unicos = df_scopus_raw.sort_values(by='Percentile', ascending=False)
df_scopus_unicos = df_scopus_unicos.drop_duplicates(subset=['Title'], keep='first').copy()

print("Etapa 2: Preparando a base de artigos do Lattes...")
# Limpeza de segurança padrão nas chaves dos artigos
df_bib_artigos['revista'] = df_bib_artigos['revista'].astype(str).str.upper().str.strip()

# Aplica a MESMA função no Lattes para garantir que ambos os lados tenham 8 dígitos no Match!
df_bib_artigos['issn'] = df_bib_artigos['issn'].apply(formatar_issn)

df_bib_artigos['doi'] = (
    df_bib_artigos['doi']
    .astype(str)
    .str.strip()
    .str.lower()
    .replace({'nan': pd.NA, 'none': pd.NA, '': pd.NA})
)

print("Etapa 3: Realizando o cruzamento exato inicial (ISSN e Nome Exato)...")
colunas_scopus = [
    'Scopus Source ID', 'Title', 'Percentile', 
    'Scopus ASJC Code (Sub-subject Area)', 'Scopus Sub-Subject Area', 'E-ISSN', 'Print ISSN'
]
df_scopus_filtro = df_scopus_unicos[colunas_scopus]

# Match pelo Nome Exato da Revista
df_match_nome = pd.merge(df_bib_artigos, df_scopus_filtro, left_on='revista', right_on='Title', how='inner')

# Match pelo E-ISSN
df_match_e_issn = pd.merge(df_bib_artigos, df_scopus_filtro, left_on='issn', right_on='E-ISSN', how='inner')

# Match pelo Print ISSN
df_match_print_issn = pd.merge(df_bib_artigos, df_scopus_filtro, left_on='issn', right_on='Print ISSN', how='inner')

# Consolida os sucessos exatos e remove possíveis duplicados
df_sucessos = pd.concat([df_match_nome, df_match_e_issn, df_match_print_issn], ignore_index=True)
df_sucessos = df_sucessos.drop_duplicates(subset=['titulo', 'id_lattes']).copy()

# Cria a coluna booleana de Computação
df_sucessos['Computation Area'] = (df_sucessos['Title'].isin(titulos_computacao) | 
                                   df_sucessos['E-ISSN'].isin(issns_computacao) | 
                                   df_sucessos['Print ISSN'].isin(issns_computacao))

print("Etapa 4: Busca Bidirecional (Nome Contido) para os artigos restantes...")
# Identifica quais artigos ainda não deram match
artigos_com_match_exato = set(df_sucessos['titulo'] + df_sucessos['id_lattes'])
df_restante = df_bib_artigos[~(df_bib_artigos['titulo'] + df_bib_artigos['id_lattes']).isin(artigos_com_match_exato)].copy()

# Prepara a lista do Scopus ordenada por tamanho do título para evitar roubo de match
df_scopus_filtro_sorted = df_scopus_filtro.copy()
df_scopus_filtro_sorted['tamanho_titulo'] = df_scopus_filtro_sorted['Title'].str.len()
df_scopus_filtro_sorted = df_scopus_filtro_sorted.sort_values(by='tamanho_titulo', ascending=False)
lista_scopus = df_scopus_filtro_sorted.to_dict('records')

# Função que busca um nome dentro do outro
def busca_bidirecional_revista(revista_lattes):
    if pd.isna(revista_lattes) or revista_lattes == 'NAN' or revista_lattes == '':
        return None

    for scopus in lista_scopus:
        titulo_scopus = scopus['Title']
        if pd.notna(titulo_scopus) and titulo_scopus != 'NAN' and titulo_scopus != "":
            # A mágica bidirecional ocorre aqui
            if (titulo_scopus in revista_lattes) or (revista_lattes in titulo_scopus):
                return scopus # Retorna o dicionário com os dados da Scopus encontrados
    return None

# Aplica a função iterativa apenas no dataframe restante (salva processamento)
resultados_parciais = df_restante['revista'].apply(busca_bidirecional_revista)

# Isola quem conseguiu match nessa etapa e os transforma em Dataframe
mask_encontrados = resultados_parciais.notna()
df_match_parcial = df_restante[mask_encontrados].copy()

if not df_match_parcial.empty:
    # Extrai as informações de Scopus do dicionário para as respectivas colunas
    dicts_encontrados = resultados_parciais[mask_encontrados]
    for col in colunas_scopus:
        df_match_parcial[col] = [d[col] for d in dicts_encontrados]
    
    # Aplica a verificação de computação
    df_match_parcial['Computation Area'] = (df_match_parcial['Title'].isin(titulos_computacao) | 
                                            df_match_parcial['E-ISSN'].isin(issns_computacao) | 
                                            df_match_parcial['Print ISSN'].isin(issns_computacao))
    
    # Junta esses novos achados à lista principal de sucessos
    df_sucessos = pd.concat([df_sucessos, df_match_parcial], ignore_index=True)
    df_sucessos = df_sucessos.drop_duplicates(subset=['titulo', 'id_lattes']).copy()

print("Etapa 5: Isolando e tratando as falhas definitivas...")
# Quem não passou nem no exato nem na busca parcial, cai aqui
df_falhas = df_restante[~mask_encontrados].copy()

# Preenche os atributos solicitados
df_falhas['Scopus Source ID'] = pd.NA
df_falhas['Title'] = pd.NA
df_falhas['Percentile'] = 0  
df_falhas['Scopus ASJC Code (Sub-subject Area)'] = pd.NA
df_falhas['Scopus Sub-Subject Area'] = pd.NA
df_falhas['E-ISSN'] = pd.NA
df_falhas['Print ISSN'] = pd.NA 
df_falhas['Computation Area'] = False  

print("Etapa 6: Consolidando a tabela final...")
# Empilha tudo (sucessos iniciais + sucessos bidirecionais + falhas)
df_artigos_final = pd.concat([df_sucessos, df_falhas], ignore_index=True)

# Garante a tipagem
df_artigos_final['Percentile'] = df_artigos_final['Percentile'].astype(int)
df_artigos_final['Computation Area'] = df_artigos_final['Computation Area'].astype(bool)

print("\n🚀 Tabela final construída com sucesso!")
print(f"Total de linhas: {len(df_artigos_final)}")

# Ordena as colunas finais
colunas_finais = [
    'id_lattes', 'titulo', 'revista', 'ano', 'doi', 'autores', # <--- 'autores' ADICIONADO AQUI
    'Scopus Source ID', 'Title', 'Percentile', 
    'Scopus ASJC Code (Sub-subject Area)', 'Scopus Sub-Subject Area', 'E-ISSN', 'Print ISSN', 
    'Computation Area' 
]
df_artigos_final = df_artigos_final[colunas_finais]

# Mostra validação
display(df_artigos_final.sample(10))

Etapa 1: Carregando e preparando a base completa da Scopus...


Etapa 2: Preparando a base de artigos do Lattes...
Etapa 3: Realizando o cruzamento exato inicial (ISSN e Nome Exato)...
Etapa 4: Busca Bidirecional (Nome Contido) para os artigos restantes...
Etapa 5: Isolando e tratando as falhas definitivas...
Etapa 6: Consolidando a tabela final...

🚀 Tabela final construída com sucesso!
Total de linhas: 2011


,id_lattes,titulo,revista,ano,doi,autores,Scopus Source ID,Title,Percentile,Scopus ASJC Code (Sub-subject Area),Scopus Sub-Subject Area,E-ISSN,Print ISSN,Computation Area
468,3957046121364560,The total chromatic number of split-indifferen...,DISCRETE MATHEMATICS,2012,http://dx.doi.org/10.1016/j.disc.2012.01.019,"CAMPOS, C. N. ; FIGUEIREDO, C. M. H. ; MELLO, ...",25892,DISCRETE MATHEMATICS,57,2607,Discrete Mathematics and Combinatorics,NaN,0012365X,True
284,2704717555047499,Functional Gradient Descent for ...,NEUROCOMPUTING,2022,http://dx.doi.org/10.1016/j.neucom.2022.05.114,"KATOPODIS, RAFAEL F. ; LIMA, PRISCILA M.V. ; F...",24807,NEUROCOMPUTING,96,2805,Cognitive Neuroscience,18728286,09252312,True
1923,1420784392366957,Online Deep Learning Hyperparameter Tuning bas...,JOURNAL OF INFORMATION AND DATA MANAGEMENT - JIDM,2021,http://dx.doi.org/10.5753/jidm.2021.1924,"KUNSTMANN, L. ; PINA, D. ; SILVA, F. ; PAES, A...",21233,MANAGEMENT,31,1400,"Business, Management and Accounting (all)",18463363,13310194,False
307,9719247117370600,Unseen: Advancing Digital Accessibility with B...,JOURNAL ON INTERACTIVE SYSTEMS,2025,http://dx.doi.org/10.5753/jis.2025.4439,"Rodrigues, C.S.C. ; NAZARETH, V. ; AZEVEDO, R....",21101250389,JOURNAL ON INTERACTIVE SYSTEMS,29,1710,Information Systems,27637719,NaN,True
994,8130520066599912,Information Technology and Productivity: Evide...,INFORMATION TECHNOLOGY FOR DEVELOPMENT,2008,NaN,"MENDONCA, M. A. A. ; Freitas, F. ; SOUZA, J. M.",14789,INFORMATION TECHNOLOGY FOR DEVELOPMENT,98,3321,Public Administration,15540170,02681102,True
1230,4602221579308599,Quasispecies dynamics on a network of interact...,JOURNAL OF STATISTICAL MECHANICS,2016,http://dx.doi.org/10.1088/1742-5468/2016/06/06...,"BARBOSA, VALMIR C.; DONANGELO, R. ; SOUZA, S. ...",145274,JOURNAL OF STATISTICAL MECHANICS: THEORY AND E...,86,2613,Statistics and Probability,17425468,NaN,False
1875,5370222318394867,Structural characterization and decomposition ...,ELECTRONIC NOTES IN DISCRETE MATHEMATICS,2015,http://dx.doi.org/10.1016/j.endm.2015.07.023,"Couto, Fernanda ; FARIA, L. ; GRAVIER, S. ; KL...",25892,DISCRETE MATHEMATICS,57,2607,Discrete Mathematics and Combinatorics,NaN,0012365X,True
211,2002515486942024,A Characterization of Edge Clique Graphs,ARS COMBINATORIA,2001,NaN,"CERIOLI, M. R. ; SZWARCFITER, J. L.",25215,ARS COMBINATORIA,4,2600,Mathematics (all),28175204,03817032,False
639,4602221579308599,Adaptive event sensing in networks of autonomo...,JOURNAL OF NETWORK AND COMPUTER APPLICATIONS,2016,http://dx.doi.org/10.1016/j.jnca.2016.04.022,"ESCH, R. R. ; PROTTI, F. ; BARBOSA, V. C.",27277,JOURNAL OF NETWORK AND COMPUTER APPLICATIONS,98,1705,Computer Networks and Communications,10958592,10848045,True
1701,6243465206463403,Dandarah sistema IoT em prol da segurança e da...,REVISTA DE SAÚDE DIGITAL E TECNOLOGIAS EDUCACI...,2021,http://dx.doi.org/10.36517/resdite.v6.n1.2021.re2,"SILVA, E. G. M. ; MATOS, B. C. ; ARAUJO, I. S....",21101187717,DIGITAL,67,1701,Computer Science (miscellaneous),26736470,NaN,True


In [24]:
# Cria a coluna indicando se o match ocorreu de forma adequada.
# A célula é idempotente: funciona tanto antes quanto depois da renomeação final.
if 'Scopus Source ID' in df_artigos_final.columns:
    coluna_id_scopus = 'Scopus Source ID'
elif 'id_scopus' in df_artigos_final.columns:
    coluna_id_scopus = 'id_scopus'
else:
    coluna_id_scopus = None

if coluna_id_scopus:
    df_artigos_final['match_adequado'] = df_artigos_final[coluna_id_scopus].notna()
else:
    df_artigos_final['match_adequado'] = False

# Atualizando a lista de colunas para exibir essa nova no começo.
# Mantém compatibilidade com a estrutura antiga e com a estrutura já renomeada.
if 'titulo' in df_artigos_final.columns:
    colunas_finais = [
        'id_lattes', 'titulo', 'revista', 'ano', 'doi', 'autores', 'match_adequado',
        'Scopus Source ID', 'Title', 'Percentile', 
        'Scopus ASJC Code (Sub-subject Area)', 'Scopus Sub-Subject Area', 'E-ISSN', 
        'Computation Area'
    ]
else:
    colunas_finais = [
        'id_lattes', 'titulo_artigo', 'titulo_revista_lattes', 'ano_pub', 'doi', 'autores',
        'match_adequado', 'id_scopus', 'titulo_revista_scopus', 'maior_percentil',
        'codigo_area_maior_percentil', 'area_maior_percentil', 'issn', 'computation_area'
    ]

colunas_finais = [col for col in colunas_finais if col in df_artigos_final.columns]
df_artigos_final = df_artigos_final[colunas_finais]

print("\nColuna 'match_adequado' adicionada com sucesso!")
if 'titulo' in df_artigos_final.columns:
    display(df_artigos_final[['titulo', 'revista', 'match_adequado', 'Percentile']].sample(10))
else:
    display(df_artigos_final[['titulo_artigo', 'titulo_revista_lattes', 'match_adequado', 'maior_percentil']].sample(10))


Coluna 'match_adequado' adicionada com sucesso!


,titulo,revista,match_adequado,Percentile
1847,Adaptive multi-resolution triangulations based...,COMMUNICATIONS IN NUMERICAL METHODS IN ENGINEE...,True,80
1357,Calculating availability and performability me...,JOURNAL OF THE ASSOCIATION FOR COMPUTING MACHI...,True,81
797,An existence result for Minty variational ineq...,ACTA MATHEMATICA VIETNAMICA,True,19
215,Generating all the Acyclic Orientations of an ...,INFORMATION PROCESSING LETTERS,True,41
81,Good and Fast Row-Sparse ah-Symmetric Reflexiv...,OPEN JOURNAL OF MATHEMATICAL OPTIMIZATION,True,67
923,How is a chordal graph like a supersolvable bi...,DISCRETE MATHEMATICS,True,57
990,Adaptative methodology of sustainability indic...,INTERNATIONAL JOURNAL OF GLOBAL ENVIRONMENTAL ...,True,26
1533,A machine learning based branch-cut-and-Bender...,TRANSPORTATION RESEARCH PART E-LOGISTICS AND T...,True,97
1406,Independent links: A new approach to increase ...,COMPUTER NETWORKS (1999),True,87
501,Stable skew partition problem,DISCRETE APPLIED MATHEMATICS,True,73


In [25]:
df_artigos_final.info()

<class 'pandas.DataFrame'>
RangeIndex: 2011 entries, 0 to 2010
Data columns (total 14 columns):
 #   Column                               Non-Null Count  Dtype 
---  ------                               --------------  ----- 
 0   id_lattes                            2011 non-null   str   
 1   titulo                               2011 non-null   str   
 2   revista                              2011 non-null   str   
 3   ano                                  2011 non-null   Int64 
 4   doi                                  1527 non-null   str   
 5   autores                              2011 non-null   str   
 6   match_adequado                       2011 non-null   bool  
 7   Scopus Source ID                     1942 non-null   object
 8   Title                                1942 non-null   object
 9   Percentile                           2011 non-null   int64 
 10  Scopus ASJC Code (Sub-subject Area)  1942 non-null   object
 11  Scopus Sub-Subject Area              1942 non-null   o

In [26]:
print("Renomeando as colunas do DataFrame final...")

# Dicionário com o mapeamento "Nome Antigo" : "Nome Novo"
mapeamento_colunas = {
    'titulo': 'titulo_artigo',
    'revista': 'titulo_revista_lattes',
    'ano': 'ano_pub',
    'Scopus Source ID': 'id_scopus',
    'Title': 'titulo_revista_scopus',
    'Percentile': 'maior_percentil',
    'Scopus ASJC Code (Sub-subject Area)': 'codigo_area_maior_percentil',
    'Scopus Sub-Subject Area': 'area_maior_percentil',
    'E-ISSN': 'issn',
    'Computation Area': 'computation_area'
}

# Aplica a renomeação diretamente no dataframe
df_artigos_final.rename(columns=mapeamento_colunas, inplace=True)

print("Colunas renomeadas com sucesso! Nova estrutura:")
df_artigos_final.info()

Renomeando as colunas do DataFrame final...
Colunas renomeadas com sucesso! Nova estrutura:
<class 'pandas.DataFrame'>
RangeIndex: 2011 entries, 0 to 2010
Data columns (total 14 columns):
 #   Column                       Non-Null Count  Dtype 
---  ------                       --------------  ----- 
 0   id_lattes                    2011 non-null   str   
 1   titulo_artigo                2011 non-null   str   
 2   titulo_revista_lattes        2011 non-null   str   
 3   ano_pub                      2011 non-null   Int64 
 4   doi                          1527 non-null   str   
 5   autores                      2011 non-null   str   
 6   match_adequado               2011 non-null   bool  
 7   id_scopus                    1942 non-null   object
 8   titulo_revista_scopus        1942 non-null   object
 9   maior_percentil              2011 non-null   int64 
 10  codigo_area_maior_percentil  1942 non-null   object
 11  area_maior_percentil         1942 non-null   object
 12  issn     

### Database de conferências

In [27]:
df_google_raw = pd.read_csv('eventos_classificados_dois_idiomas.csv')

In [28]:
df_google_raw.head(3)

,Sigla,Nome do evento em inglês,Nome do evento,Estrato
0,AAAI,AAAI Conference on Artificial Intelligence,Conferência AAAI sobre Inteligência Artificial,A1
1,AAMAS,International Conference on Autonomous Agents ...,Conferência Internacional sobre Agentes Autôno...,A1
2,ACCV,Asian Conference on Computer Vision,Conferência Asiática sobre Visão Computacional,A1


In [29]:
df_google_raw.info()

<class 'pandas.DataFrame'>
RangeIndex: 781 entries, 0 to 780
Data columns (total 4 columns):
 #   Column                    Non-Null Count  Dtype
---  ------                    --------------  -----
 0   Sigla                     781 non-null    str  
 1   Nome do evento em inglês  781 non-null    str  
 2   Nome do evento            781 non-null    str  
 3   Estrato                   781 non-null    str  
dtypes: str(4)
memory usage: 124.6 KB


In [30]:
df_google_raw['Estrato'].value_counts()

Estrato
A3    171
A4    134
A1    110
B4     90
A2     86
B1     78
B2     60
B3     52
Name: count, dtype: int64

In [31]:
print("Substituindo as classificações na coluna 'Estrato'...")

# 1. Cria o dicionário de substituição ('Valor Antigo': 'Valor Novo')
mapeamento_estratos = {
    'B1': 'A5',
    'B2': 'A6',
    'B3': 'A7',
    'B4': 'A8'
}

# 2. Aplica a substituição apenas na coluna 'Estrato'
df_google_raw['Estrato'] = df_google_raw['Estrato'].replace(mapeamento_estratos)

# 3. Verifica o resultado para garantir que deu certo
print("\nNova distribuição de Estratos:")
display(df_google_raw['Estrato'].value_counts())

Substituindo as classificações na coluna 'Estrato'...

Nova distribuição de Estratos:


Estrato
A3    171
A4    134
A1    110
A8     90
A2     86
A5     78
A6     60
A7     52
Name: count, dtype: int64

In [32]:
# Transforma a coluna 'Nome do evento' para letras maiúsculas
df_google_raw['Nome do evento'] = df_google_raw['Nome do evento'].str.upper()

# Exibe as primeiras linhas para confirmar a alteração
display(df_google_raw.head())

,Sigla,Nome do evento em inglês,Nome do evento,Estrato
0,AAAI,AAAI Conference on Artificial Intelligence,CONFERÊNCIA AAAI SOBRE INTELIGÊNCIA ARTIFICIAL,A1
1,AAMAS,International Conference on Autonomous Agents ...,CONFERÊNCIA INTERNACIONAL SOBRE AGENTES AUTÔNO...,A1
2,ACCV,Asian Conference on Computer Vision,CONFERÊNCIA ASIÁTICA SOBRE VISÃO COMPUTACIONAL,A1
3,ACII,International Conference on Affective Computin...,CONFERÊNCIA INTERNACIONAL SOBRE COMPUTAÇÃO AFE...,A2
4,ACISP,Australasian Conference on Information Securit...,CONFERÊNCIA AUSTRALASIÁTICA SOBRE SEGURANÇA E ...,A4


In [33]:
import pandas as pd
import re

print("1. Preparando dados, removendo acentos e ordenando por tamanho do nome...")

# Cria uma função encadeada para limpar o texto: remove acentos, joga para maiúsculo e tira espaços nas bordas
def limpar_texto(serie):
    return (serie.astype(str)
            .str.normalize('NFKD')
            .str.encode('ascii', errors='ignore')
            .str.decode('utf-8')
            .str.upper()
            .str.replace(r'\s+', ' ', regex=True) # <- Escudo contra espaços múltiplos e invisíveis
            .str.strip())                     

# Aplica a limpeza nas bases
df_bib_trab_congresso['evento_limpo'] = limpar_texto(df_bib_trab_congresso['evento'])
df_google_raw['Nome do evento'] = limpar_texto(df_google_raw['Nome do evento'])
df_google_raw['Nome do evento em inglês'] = limpar_texto(df_google_raw['Nome do evento em inglês']) # Nova coluna
df_google_raw['Sigla'] = limpar_texto(df_google_raw['Sigla'])

# O truque de ouro atualizado: encontrar o maior nome entre PT e EN para ordenar
# Isso evita que um nome curto em uma das línguas roube o match de um nome longo na outra
df_google_raw['tamanho_pt'] = df_google_raw['Nome do evento'].str.len()
df_google_raw['tamanho_en'] = df_google_raw['Nome do evento em inglês'].str.len()
df_google_raw['tamanho_max'] = df_google_raw[['tamanho_pt', 'tamanho_en']].max(axis=1)

# Ordena pelo maior nome disponível
df_google_raw = df_google_raw.sort_values(by='tamanho_max', ascending=False)

# Transforma a base do Google em uma lista de dicionários para a busca ser ultrarrápida
lista_google = df_google_raw.to_dict('records')

print("2. Aplicando a lógica de match (Nomes PT/EN Bidirecional -> Sigla)...")

def encontrar_melhor_match(evento_lattes):
    if pd.isna(evento_lattes) or evento_lattes == 'NAN' or evento_lattes == '':
        return pd.NA, pd.NA, 'A8', 'Sem Match'

    # Tentativa 1: Verifica de forma BIDIRECIONAL em ambas as colunas (PT e EN)
    for google in lista_google:
        nome_pt = google['Nome do evento']
        nome_en = google['Nome do evento em inglês']
        
        # Checa a versão em Português
        if pd.notna(nome_pt) and nome_pt != 'NAN' and nome_pt != "":
            if (nome_pt in evento_lattes) or (evento_lattes in nome_pt):
                # Retorna sempre o nome em PT como principal para padronizar a tabela final
                return google['Sigla'], google['Nome do evento'], google['Estrato'], 'Por Nome PT'

        # Checa a versão em Inglês
        if pd.notna(nome_en) and nome_en != 'NAN' and nome_en != "":
            if (nome_en in evento_lattes) or (evento_lattes in nome_en):
                # Retorna sempre o nome em PT como principal para padronizar a tabela final
                return google['Sigla'], google['Nome do evento'], google['Estrato'], 'Por Nome EN'

    # Tentativa 2: Se falhar nos nomes, busca a Sigla protegida por limites de palavra (\b)
    for google in lista_google:
        sigla = google['Sigla']
        if pd.notna(sigla) and sigla != 'NAN' and sigla != "":
            padrao = r'\b' + re.escape(sigla) + r'\b'
            if re.search(padrao, evento_lattes):
                return google['Sigla'], google['Nome do evento'], google['Estrato'], 'Por Sigla'

    return pd.NA, pd.NA, 'A8', 'Sem Match'

print("Isso pode levar alguns segundos...")
# Aplica a função de busca
resultados = df_bib_trab_congresso['evento_limpo'].apply(encontrar_melhor_match)

print("3. Consolidando base final...")
df_artigos_congresso_final = df_bib_trab_congresso.copy()

# Extraindo os resultados da função para suas respectivas colunas
df_artigos_congresso_final['Sigla'] = [res[0] for res in resultados]
df_artigos_congresso_final['Nome do evento'] = [res[1] for res in resultados]
df_artigos_congresso_final['Estrato'] = [res[2] for res in resultados]
df_artigos_congresso_final['tipo_match'] = [res[3] for res in resultados]

# Remove colunas auxiliares de limpeza
df_artigos_congresso_final.drop(columns=['evento_limpo'], inplace=True)

# ==========================================
# RESUMO DOS RESULTADOS
# ==========================================
total_originais = len(df_artigos_congresso_final)
qtd_nome_pt = len(df_artigos_congresso_final[df_artigos_congresso_final['tipo_match'] == 'Por Nome PT'])
qtd_nome_en = len(df_artigos_congresso_final[df_artigos_congresso_final['tipo_match'] == 'Por Nome EN'])
qtd_sigla = len(df_artigos_congresso_final[df_artigos_congresso_final['tipo_match'] == 'Por Sigla'])
qtd_falhas = len(df_artigos_congresso_final[df_artigos_congresso_final['tipo_match'] == 'Sem Match'])

print(f"\n--- 📊 RELATÓRIO DE CRUZAMENTO DE EVENTOS ---")
print(f"Total de Artigos (Lattes): {total_originais}")
print(f"✅ Match por Nome PT: {qtd_nome_pt} ({round((qtd_nome_pt/total_originais)*100, 1)}%)")
print(f"✅ Match por Nome EN: {qtd_nome_en} ({round((qtd_nome_en/total_originais)*100, 1)}%)")
print(f"✅ Match por Sigla: {qtd_sigla} ({round((qtd_sigla/total_originais)*100, 1)}%)")
print(f"❌ Sem Match: {qtd_falhas} ({round((qtd_falhas/total_originais)*100, 1)}%)")

# Exibe uma amostra dos matches para validação
display(df_artigos_congresso_final[df_artigos_congresso_final['tipo_match'] != 'Sem Match'][['evento', 'Nome do evento', 'Estrato', 'tipo_match']].sample(5))

1. Preparando dados, removendo acentos e ordenando por tamanho do nome...
2. Aplicando a lógica de match (Nomes PT/EN Bidirecional -> Sigla)...
Isso pode levar alguns segundos...
3. Consolidando base final...

--- 📊 RELATÓRIO DE CRUZAMENTO DE EVENTOS ---
Total de Artigos (Lattes): 3670
✅ Match por Nome PT: 138 (3.8%)
✅ Match por Nome EN: 990 (27.0%)
✅ Match por Sigla: 844 (23.0%)
❌ Sem Match: 1698 (46.3%)


,evento,Nome do evento,Estrato,tipo_match
1885,ENCONTRO NACIONAL DE INTELIGENCIA ARTIFICIAL E...,NATIONAL MEETING ON ARTIFICIAL AND COMPUTATION...,A4,Por Nome EN
2388,11TH INTERNATIONAL NETWORK OPTIMIZATION CONFER...,CONFERENCIA INTERNACIONAL DE OTIMIZACAO DE REDES,A7,Por Nome EN
3583,1ST INTERNATIONAL WORKSHOP ON BIOLOGICAL DATA ...,CONFERENCIA INTERNACIONAL SOBRE CIENCIA DE DAD...,A1,Por Sigla
3189,THE 13TH IEEE INTERNATIONAL CONFERENCE ON CSCW...,CONFERENCIA INTERNACIONAL IEEE SOBRE TRABALHO ...,A3,Por Sigla
943,XX SIMPÓSIO BRASILEIRO DE ENGENHARIA DE SOFTWARE,SIMPOSIO BRASILEIRO DE ENGENHARIA DE SOFTWARE,A3,Por Nome PT


In [34]:
import pandas as pd
import re
from rapidfuzz import fuzz

# [Seu Passo 1 (limpar_texto) continua exatamente igual aqui]

print("2. Aplicando a lógica Fuzzy de Alta Precisão (Nomes PT/EN -> Sigla)...")

# ==========================================
# CONFIGURAÇÃO DE RIGOR (0 a 100)
# ==========================================
# 90 é um valor excelente para evitar falsos positivos. 
# Só vai dar match se as palavras principais forem praticamente idênticas.
LIMIAR_CORTE_FUZZY = 95 

def encontrar_melhor_match_fuzzy(evento_lattes):
    if pd.isna(evento_lattes) or evento_lattes == 'NAN' or evento_lattes == '':
        return pd.NA, pd.NA, 'A8', 'Sem Match', 0

    melhor_google_match = None
    maior_score_encontrado = 0
    tipo_do_melhor_match = 'Sem Match'

    # ---------------------------------------------------------
    # TENTATIVA 1: Busca Exata pela Sigla (Altíssima Confiança)
    # ---------------------------------------------------------
    for google in lista_google:
        sigla = google['Sigla']
        if pd.notna(sigla) and sigla != 'NAN' and sigla != "":
            padrao = r'\b' + re.escape(sigla) + r'\b'
            if re.search(padrao, evento_lattes):
                # Se achou a sigla isolada, é match imediato (Score 100)
                return google['Sigla'], google['Nome do evento'], google['Estrato'], 'Por Sigla Exata', 100

    # ---------------------------------------------------------
    # TENTATIVA 2: Busca Fuzzy usando Token Set Ratio
    # ---------------------------------------------------------
    for google in lista_google:
        nome_pt = google['Nome do evento']
        nome_en = google['Nome do evento em inglês']
        
        score_pt = 0
        score_en = 0

        # Calcula a similaridade do conjunto de palavras em PT
        if pd.notna(nome_pt) and nome_pt != 'NAN' and nome_pt != "":
            score_pt = fuzz.token_set_ratio(evento_lattes, nome_pt)
            
        # Calcula a similaridade do conjunto de palavras em EN
        if pd.notna(nome_en) and nome_en != 'NAN' and nome_en != "":
            score_en = fuzz.token_set_ratio(evento_lattes, nome_en)

        # Pega o melhor score entre a versão PT e EN para este evento
        score_atual_max = max(score_pt, score_en)

        # Atualiza se encontrarmos um score melhor do que os anteriores
        if score_atual_max > maior_score_encontrado:
            maior_score_encontrado = score_atual_max
            melhor_google_match = google
            tipo_do_melhor_match = 'Fuzzy Nome PT' if score_pt >= score_en else 'Fuzzy Nome EN'

    # ---------------------------------------------------------
    # DECISÃO FINAL: O melhor match supera nosso limiar de segurança?
    # ---------------------------------------------------------
    if maior_score_encontrado >= LIMIAR_CORTE_FUZZY:
        return (
            melhor_google_match['Sigla'], 
            melhor_google_match['Nome do evento'], 
            melhor_google_match['Estrato'], 
            tipo_do_melhor_match, 
            maior_score_encontrado
        )

    # Se o melhor score foi menor que o limiar, titulo_evento_lattes, rejeitamos (evita falso positivo)
    return pd.NA, pd.NA, 'A8', 'Sem Match', maior_score_encontrado

print("Isso pode levar alguns segundos...")
# Aplica a função de busca
resultados = df_bib_trab_congresso['evento_limpo'].apply(encontrar_melhor_match_fuzzy)

print("3. Consolidando base final...")
df_artigos_congresso_final = df_bib_trab_congresso.copy()

# Extraindo os resultados, agora incluindo a coluna de Score para sua auditoria
df_artigos_congresso_final['Sigla'] = [res[0] for res in resultados]
df_artigos_congresso_final['Nome do evento'] = [res[1] for res in resultados]
df_artigos_congresso_final['Estrato'] = [res[2] for res in resultados]
df_artigos_congresso_final['tipo_match'] = [res[3] for res in resultados]
df_artigos_congresso_final['score_confianca'] = [res[4] for res in resultados]

df_artigos_congresso_final.drop(columns=['evento_limpo'], inplace=True)

2. Aplicando a lógica Fuzzy de Alta Precisão (Nomes PT/EN -> Sigla)...
Isso pode levar alguns segundos...
3. Consolidando base final...


In [35]:
import pandas as pd

# 1. Definindo a "Zona Crítica"
# São os matches que passaram do limiar de corte, mas não são 100% idênticos
# Ajuste o nome do DataFrame para o seu de periódicos, se for o caso
limiar_inferior = 95  # O valor que definimos no código anterior
limiar_superior = 99  # Tudo abaixo de 100 (100 = match exato ou sigla perfeita)

df_zona_critica = df_artigos_congresso_final[
    (df_artigos_congresso_final['score_confianca'] >= limiar_inferior) & 
    (df_artigos_congresso_final['score_confianca'] <= limiar_superior)
].copy()

# 2. Ordenando pelo maior risco (scores mais baixos primeiro)
df_zona_critica = df_zona_critica.sort_values(by='score_confianca', ascending=True)

# 3. Selecionando apenas as colunas que importam para o "tira-teima" visual
# (Ajuste os nomes das colunas de acordo com a sua tabela de periódicos)
colunas_para_auditoria = [
    'evento',           # Nome original que veio do Lattes do pesquisador
    'Nome do evento',   # Nome que o algoritmo puxou da base de qualificação
    'Sigla',
    'Estrato', 
    'tipo_match',
    'score_confianca'
]

tabela_auditoria = df_zona_critica[colunas_para_auditoria]

# 4. Exibindo os resultados e aplicando estilo visual (se estiver no Jupyter/Colab)
print(f"⚠️ ATENÇÃO: Encontrados {len(tabela_auditoria)} registros na Zona Crítica (Score {limiar_inferior} a {limiar_superior}).")
print("Recomenda-se leitura atenta para garantir que não há homônimos.\n")

# Se você estiver usando Jupyter Notebook ou Google Colab, a linha abaixo 
# cria uma tabela com um gradiente de cores na coluna de score para facilitar a visão.
display(
    tabela_auditoria.style.background_gradient(
        subset=['score_confianca'], 
        cmap='YlOrRd_r', # Vermelho para 90, Amarelo para 99
        vmin=limiar_inferior, 
        vmax=limiar_superior
    )
)

⚠️ ATENÇÃO: Encontrados 124 registros na Zona Crítica (Score 95 a 99).
Recomenda-se leitura atenta para garantir que não há homônimos.



,evento,Nome do evento,Sigla,Estrato,tipo_match,score_confianca
499,XIX SIMPÓSIO BRASILEIRO DE TELECOMUNICAÇÕES,BRAZILIAN SYMPOSIUM ON TELECOMMUNICATIONS AND SIGNAL PROCESSING,SBRT,A8,Fuzzy Nome EN,95.121951
1156,XXI SIMPÓSIO BRASILEIRO DE TELECOMUNICAÇÕES,BRAZILIAN SYMPOSIUM ON TELECOMMUNICATIONS AND SIGNAL PROCESSING,SBRT,A8,Fuzzy Nome EN,95.121951
1158,XXI SIMPÓSIO BRASILEIRO DE TELECOMUNICAÇÕES,BRAZILIAN SYMPOSIUM ON TELECOMMUNICATIONS AND SIGNAL PROCESSING,SBRT,A8,Fuzzy Nome EN,95.121951
1159,XXI SIMPÓSIO BRASILEIRO DE TELECOMUNICAÇÕES,BRAZILIAN SYMPOSIUM ON TELECOMMUNICATIONS AND SIGNAL PROCESSING,SBRT,A8,Fuzzy Nome EN,95.121951
1165,XXI SIMPÓSIO BRASILEIRO DE TELECOMUNICAÇÕES,BRAZILIAN SYMPOSIUM ON TELECOMMUNICATIONS AND SIGNAL PROCESSING,SBRT,A8,Fuzzy Nome EN,95.121951
1166,XXI SIMPÓSIO BRASILEIRO DE TELECOMUNICAÇÕES,BRAZILIAN SYMPOSIUM ON TELECOMMUNICATIONS AND SIGNAL PROCESSING,SBRT,A8,Fuzzy Nome EN,95.121951
1167,XXI SIMPÓSIO BRASILEIRO DE TELECOMUNICAÇÕES,BRAZILIAN SYMPOSIUM ON TELECOMMUNICATIONS AND SIGNAL PROCESSING,SBRT,A8,Fuzzy Nome EN,95.121951
1168,XXI SIMPÓSIO BRASILEIRO DE TELECOMUNICAÇÕES,BRAZILIAN SYMPOSIUM ON TELECOMMUNICATIONS AND SIGNAL PROCESSING,SBRT,A8,Fuzzy Nome EN,95.121951
1147,XXI SIMPÓSIO BRASILEIRO DE TELECOMUNICAÇÕES,BRAZILIAN SYMPOSIUM ON TELECOMMUNICATIONS AND SIGNAL PROCESSING,SBRT,A8,Fuzzy Nome EN,95.121951
1431,XXI SIMPÓSIO BRASILEIRO DE TELECOMUNICAÇÕES,BRAZILIAN SYMPOSIUM ON TELECOMMUNICATIONS AND SIGNAL PROCESSING,SBRT,A8,Fuzzy Nome EN,95.121951


In [36]:
df_artigos_congresso_final.head()
print(df_artigos_congresso_final.info())

<class 'pandas.DataFrame'>
RangeIndex: 3670 entries, 0 to 3669
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   titulo           3670 non-null   str    
 1   ano              3670 non-null   Int64  
 2   doi              1053 non-null   str    
 3   autores          3670 non-null   str    
 4   evento           3658 non-null   str    
 5   cidade           0 non-null      str    
 6   paginas          2571 non-null   str    
 7   isbn             0 non-null      str    
 8   id_lattes        3670 non-null   str    
 9   Sigla            2169 non-null   str    
 10  Nome do evento   2169 non-null   str    
 11  Estrato          3670 non-null   str    
 12  tipo_match       3670 non-null   str    
 13  score_confianca  3670 non-null   float64
dtypes: Int64(1), float64(1), str(12)
memory usage: 1.4 MB
None


In [37]:
import pandas as pd

print("Padronizando os nomes das colunas de eventos...")

# Dicionário com o mapeamento "Nome Antigo" : "Nome Novo"
mapeamento_colunas_eventos = {
    'titulo': 'titulo_artigo',
    'evento': 'titulo_evento_lattes',
    'Sigla': 'sigla_evento_google',
    'Nome do evento': 'titulo_evento_google',
    'Estrato': 'estrato'
}

# Aplica a renomeação diretamente no dataframe
df_artigos_congresso_final.rename(columns=mapeamento_colunas_eventos, inplace=True)

print("Colunas renomeadas com sucesso! Nova estrutura:")
df_artigos_congresso_final.info()

Padronizando os nomes das colunas de eventos...
Colunas renomeadas com sucesso! Nova estrutura:
<class 'pandas.DataFrame'>
RangeIndex: 3670 entries, 0 to 3669
Data columns (total 14 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   titulo_artigo         3670 non-null   str    
 1   ano                   3670 non-null   Int64  
 2   doi                   1053 non-null   str    
 3   autores               3670 non-null   str    
 4   titulo_evento_lattes  3658 non-null   str    
 5   cidade                0 non-null      str    
 6   paginas               2571 non-null   str    
 7   isbn                  0 non-null      str    
 8   id_lattes             3670 non-null   str    
 9   sigla_evento_google   2169 non-null   str    
 10  titulo_evento_google  2169 non-null   str    
 11  estrato               3670 non-null   str    
 12  tipo_match            3670 non-null   str    
 13  score_confianca       3670 non-null   

In [38]:
print("Removendo as colunas 'cidade' e 'isbn'...")

# O parâmetro errors='ignore' é uma trava de segurança. 
# Se você rodar a célula duas vezes sem querer, ele não vai dar erro reclamando que a coluna já sumiu.
df_artigos_congresso_final.drop(columns=['cidade', 'isbn'], inplace=True, errors='ignore')

print("Colunas removidas com sucesso! Estrutura atualizada:")
df_artigos_congresso_final.info()

Removendo as colunas 'cidade' e 'isbn'...
Colunas removidas com sucesso! Estrutura atualizada:
<class 'pandas.DataFrame'>
RangeIndex: 3670 entries, 0 to 3669
Data columns (total 12 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   titulo_artigo         3670 non-null   str    
 1   ano                   3670 non-null   Int64  
 2   doi                   1053 non-null   str    
 3   autores               3670 non-null   str    
 4   titulo_evento_lattes  3658 non-null   str    
 5   paginas               2571 non-null   str    
 6   id_lattes             3670 non-null   str    
 7   sigla_evento_google   2169 non-null   str    
 8   titulo_evento_google  2169 non-null   str    
 9   estrato               3670 non-null   str    
 10  tipo_match            3670 non-null   str    
 11  score_confianca       3670 non-null   float64
dtypes: Int64(1), float64(1), str(10)
memory usage: 1.3 MB


### Tratando alunos

In [39]:
import unicodedata
import re
import json
import pandas as pd
from pathlib import Path

print("Tratando dados de alunos e marcando coautoria nas produções...")

# ---------------------------------------------------------
# 1. FUNÇÕES DE NORMALIZAÇÃO E GERAÇÃO DE PERMUTAÇÕES
# ---------------------------------------------------------
def normalizar_texto(valor):
    """Normaliza texto para comparação de nomes em autoria."""
    if pd.isna(valor):
        return ""

    texto = unicodedata.normalize('NFKD', str(valor))
    texto = texto.encode('ascii', errors='ignore').decode('utf-8')
    texto = texto.upper()
    texto = re.sub(r'[^A-Z0-9]+', ' ', texto)
    return re.sub(r'\s+', ' ', texto).strip()


def gerar_todas_abreviacoes(nome_completo_norm):
    """
    Gera exaustivamente todas as permutações acadêmicas de um nome normalizado,
    cobrindo variações de sobrenomes intermediários (muito comum no Brasil),
    sobrenomes compostos e nomes de batismo duplos.
    """
    preposicoes = {'DE', 'DA', 'DO', 'DAS', 'DOS', 'E'}
    partes_originais = nome_completo_norm.split()
    
    # Remove preposições para não virarem iniciais
    partes_uteis = [p for p in partes_originais if p not in preposicoes]

    if len(partes_uteis) < 2:
        return {nome_completo_norm}

    variacoes = set([nome_completo_norm])

    # 1. Define os possíveis blocos de "Sobrenome de Citação"
    sobrenomes_alvo = []
    
    # Adiciona o último nome (ex: CARNEIRO)
    sobrenomes_alvo.append(partes_uteis[-1])
    
    # Adiciona QUALQUER nome do meio como possível sobrenome principal (ex: MAUES, DIAS)
    # Isso resolve o problema de alunos que assinam com o sobrenome materno
    for i in range(1, len(partes_uteis) - 1):
        sobrenomes_alvo.append(partes_uteis[i])

    # Adiciona combinações compostas (ex: DIAS CARNEIRO)
    if len(partes_uteis) >= 3:
        sobrenomes_alvo.append(f"{partes_uteis[-2]} {partes_uteis[-1]}")

    # Adiciona com a preposição original, se houver (ex: DE CARNEIRO)
    if len(partes_originais) >= 2 and partes_originais[-2] in preposicoes:
        sobrenomes_alvo.append(f"{partes_originais[-2]} {partes_originais[-1]}")

    # 2. Combina cada possibilidade de sobrenome com as iniciais do resto do nome
    for sobrenome in set(sobrenomes_alvo):
        sobrenome_partes = sobrenome.split()
        resto = [p for p in partes_uteis if p not in sobrenome_partes]

        if not resto:
            continue

        iniciais = [p[0] for p in resto]
        primeiro_nome = resto[0]
        inicial_primeira = iniciais[0]

        iniciais_com_espaco = " ".join(iniciais)
        iniciais_sem_espaco = "".join(iniciais)
        
        # Trata nomes de batismo compostos (ex: João Vítor -> "J V")
        duas_iniciais = f"{iniciais[0]} {iniciais[1]}" if len(iniciais) > 1 else inicial_primeira

        primeiro_mais_iniciais = primeiro_nome
        if len(iniciais) > 1:
            primeiro_mais_iniciais += " " + " ".join(iniciais[1:])

        # Blocos que representam os nomes de batismo (antes do sobrenome)
        blocos_nome = [
            iniciais_com_espaco,       # J V D C
            iniciais_sem_espaco,       # JVDC
            inicial_primeira,          # J (Vai formar "MAUES J")
            duas_iniciais,             # J V (Vai formar "MAUES J V")
            primeiro_nome,             # JOAO
            primeiro_mais_iniciais,    # JOAO V D C
            " ".join(resto)            # JOAO VITOR DIAS CARNEIRO
        ]

        # 3. Faz a permutação da Ordem (Sobrenome Nome vs Nome Sobrenome)
        for bloco in set(blocos_nome):
            variacoes.add(f"{sobrenome} {bloco}")
            variacoes.add(f"{bloco} {sobrenome}")

    return variacoes

# ---------------------------------------------------------
# 2. EXTRAÇÃO E ACHATAMENTO DOS ALUNOS
# ---------------------------------------------------------
def extrair_alunos(diretorio_alunos):
    """Lê os JSONs brutos dos alunos e cria um DataFrame com expansão exaustiva."""
    registros = []
    pasta = Path(diretorio_alunos)

    if not pasta.exists() or not pasta.is_dir():
        print(f"AVISO: diretório de alunos não encontrado: {diretorio_alunos}")
        return pd.DataFrame()

    for arquivo_json in sorted(pasta.glob('*.json')):
        try:
            with open(arquivo_json, 'r', encoding='utf-8') as f:
                dados_aluno = json.load(f)

            info = dados_aluno.get('informacoes_pessoais', {})
            id_lattes = info.get('id_lattes')
            nome_completo = info.get('nome_completo', '')
            nome_citacoes = info.get('nome_citacoes', '')

            if not id_lattes or not nome_completo:
                continue

            citacoes = [item.strip() for item in str(nome_citacoes).split(';') if item.strip()]
            variacoes_normalizadas = []

            # 1. Normaliza o nome completo do aluno
            nome_comp_norm = normalizar_texto(nome_completo)
            
            # 2. Chama a geração exaustiva de variações
            todas_permutacoes = gerar_todas_abreviacoes(nome_comp_norm)
            
            for perm in todas_permutacoes:
                variacoes_normalizadas.append(perm)

            # 3. Garante que as citações originais do Lattes também entrem
            for variacao in citacoes:
                nome_norm_citacao = normalizar_texto(variacao)
                if nome_norm_citacao and nome_norm_citacao not in variacoes_normalizadas:
                    variacoes_normalizadas.append(nome_norm_citacao)

            registros.append({
                'id_lattes': str(id_lattes),
                'nome_completo': str(nome_completo).strip(),
                'nome_citacoes': str(nome_citacoes).strip(),
                'nome_completo_normalizado': nome_comp_norm,
                'nome_citacoes_normalizadas': ' | '.join(variacoes_normalizadas),
                'variacoes_coautoria': ' | '.join(variacoes_normalizadas),
            })
        except (json.JSONDecodeError, OSError) as erro:
            print(f"Erro ao ler {arquivo_json}: {erro}")

    df_alunos_local = pd.DataFrame(registros)
    if not df_alunos_local.empty:
        df_alunos_local = df_alunos_local.drop_duplicates(subset=['id_lattes']).copy()

    return df_alunos_local


df_alunos = extrair_alunos('dados_brutos/alunos')
print(f"Total de alunos carregados: {len(df_alunos)}")

# ---------------------------------------------------------
# 3. DETECÇÃO DE COAUTORIA (BUSCA GLOBAL POR SUBSTRING)
# ---------------------------------------------------------
def montar_lista_variacoes(df_alunos_local):
    """Transforma o DataFrame de alunos em conjuntos de nomes normalizados."""
    variacoes = []

    if df_alunos_local.empty:
        return variacoes

    for _, linha in df_alunos_local.iterrows():
        nomes = [item.strip() for item in str(linha.get('variacoes_coautoria', '')).split('|') if item.strip()]
        if nomes:
            variacoes.append(set(nomes))

    return variacoes


def tem_coautoria_aluno(autores, variacoes_alunos):
    """Verifica se a string contínua de autores contém alguma variação do aluno."""
    if pd.isna(autores) or not str(autores).strip() or not variacoes_alunos:
        return False

    autores_normalizados = f" {normalizar_texto(autores)} "
    
    for variacoes in variacoes_alunos:
        for nome_normalizado in variacoes:
            if f" {nome_normalizado} " in autores_normalizados:
                return True

    return False


# Se a célula for reexecutada depois da etapa de renomeação, recupera os autores dos periódicos.
if 'autores' not in df_artigos_final.columns and 'df_sucessos' in globals() and {'titulo', 'autores', 'id_lattes'}.issubset(df_sucessos.columns):
    df_artigos_final = df_artigos_final.merge(
        df_sucessos[['id_lattes', 'titulo', 'autores']].drop_duplicates(),
        left_on=['id_lattes', 'titulo_artigo'] if 'titulo_artigo' in df_artigos_final.columns else ['id_lattes', 'titulo'],
        right_on=['id_lattes', 'titulo'],
        how='left'
    )
    if 'titulo' in df_artigos_final.columns:
        df_artigos_final = df_artigos_final.drop(columns=['titulo'])


variacoes_alunos = montar_lista_variacoes(df_alunos)

for df_prod in [df_artigos_final, df_artigos_congresso_final]:
    if 'autores' in df_prod.columns:
        df_prod['coautoria_aluno'] = df_prod['autores'].apply(lambda autores: tem_coautoria_aluno(autores, variacoes_alunos))
    else:
        df_prod['coautoria_aluno'] = False

print("Coautoria marcada nas bases de periódicos e conferências.")
print(f"Periódicos com coautoria de aluno: {int(df_artigos_final['coautoria_aluno'].sum())}")
print(f"Conferências com coautoria de aluno: {int(df_artigos_congresso_final['coautoria_aluno'].sum())}")

display(df_alunos.head())

Tratando dados de alunos e marcando coautoria nas produções...
Total de alunos carregados: 297
Coautoria marcada nas bases de periódicos e conferências.
Periódicos com coautoria de aluno: 844
Conferências com coautoria de aluno: 1912


,id_lattes,nome_completo,nome_citacoes,nome_completo_normalizado,nome_citacoes_normalizadas,variacoes_coautoria
0,1766965412894981,João Luís da Silva Guio Soares,"SOARES, J. L. S. G.;GUIO, J. L.",JOAO LUIS DA SILVA GUIO SOARES,JOAO LUIS | SOARES JOAO | GUIO JOAO | GUIO SOA...,JOAO LUIS | SOARES JOAO | GUIO JOAO | GUIO SOA...
1,0352188533423371,David Ventura Cardoso,"CARDOSO, D. V.",DAVID VENTURA CARDOSO,VENTURA CARDOSO DAVID | DAVID VENTURA CARDOSO ...,VENTURA CARDOSO DAVID | DAVID VENTURA CARDOSO ...
2,8773283315440616,Ana Clara Correa da Silva,"SILVA, A. C. C.",ANA CLARA CORREA DA SILVA,DA SILVA A C C | CORREA SILVA ANA | SILVA A C ...,DA SILVA A C C | CORREA SILVA ANA | SILVA A C ...
3,6910314996365495,Fabio Luiz Silva Nogueira,"NOGUEIRA, F. L. S.",FABIO LUIZ SILVA NOGUEIRA,FABIO L S NOGUEIRA | NOGUEIRA FABIO LUIZ SILVA...,FABIO L S NOGUEIRA | NOGUEIRA FABIO LUIZ SILVA...
4,2636706873331793,Felipe Bevilaqua Foldes Guimarães,"GUIMARÃES, F. B. F.;GUIMARÃES, FELIPE BEVILAQU...",FELIPE BEVILAQUA FOLDES GUIMARAES,FELIPE FOLDES | BEVILAQUA FELIPE F G | FELIPE ...,FELIPE FOLDES | BEVILAQUA FELIPE F G | FELIPE ...


In [40]:
# import json
# import re
# import unicodedata
# from pathlib import Path

# import pandas as pd

# print("ETAPA 1: Lendo alunos e extraindo papers brutos...")


# def normalizar_texto(valor):
#     """Normaliza texto para comparações exatas por título."""
#     if pd.isna(valor):
#         return ""

#     texto = unicodedata.normalize('NFKD', str(valor))
#     texto = texto.encode('ascii', errors='ignore').decode('utf-8')
#     texto = texto.upper()
#     texto = re.sub(r'[^A-Z0-9]+', ' ', texto)
#     return re.sub(r'\s+', ' ', texto).strip()


# def normalizar_doi(valor):
#     """Normaliza DOI para comparações exatas."""
#     if pd.isna(valor):
#         return ""

#     texto = str(valor).strip().lower()
#     if texto in {'', 'nan', 'none', 'nat'}:
#         return ""

#     texto = re.sub(r'^https?://(dx\.)?doi\.org/', '', texto)
#     texto = re.sub(r'^doi:\s*', '', texto)
#     return texto.strip()


# def extrair_papers_da_lista(
#     papers,
#     *,
#     tipo_paper,
#     id_lattes_aluno,
#     nome_completo_aluno,
# ):
#     """Explode a lista de papers brutos em linhas normalizadas."""
#     registros = []

#     for paper in papers or []:
#         titulo_original = paper.get('titulo', '')
#         doi_original = paper.get('doi', '')

#         registros.append({
#             'id_lattes_aluno': id_lattes_aluno,
#             'nome_completo_aluno': nome_completo_aluno,
#             'tipo_paper': tipo_paper,
#             'titulo_paper_original': titulo_original,
#             'doi_paper_original': doi_original,
#             'titulo_key': normalizar_texto(titulo_original),
#             'doi_key': normalizar_doi(doi_original),
#             'ano_paper': paper.get('ano'),
#             'veiculo_paper': paper.get('revista') or paper.get('evento') or paper.get('nome_evento') or '',
#         })

#     return registros


# def extrair_alunos_e_papers(diretorio_alunos):
#     """Lê os JSONs dos alunos e gera df_alunos + df_papers_alunos."""
#     registros_alunos = []
#     registros_papers = []
#     pasta = Path(diretorio_alunos)

#     if not pasta.exists() or not pasta.is_dir():
#         print(f"AVISO: diretório de alunos não encontrado: {diretorio_alunos}")
#         return pd.DataFrame(), pd.DataFrame()

#     for arquivo_json in sorted(pasta.glob('*.json')):
#         try:
#             with open(arquivo_json, 'r', encoding='utf-8') as f:
#                 dados_aluno = json.load(f)
#         except (json.JSONDecodeError, OSError) as erro:
#             print(f"Erro ao ler {arquivo_json}: {erro}")
#             continue

#         info = dados_aluno.get('informacoes_pessoais', {})
#         id_lattes = info.get('id_lattes')
#         nome_completo = info.get('nome_completo', '')
#         nome_citacoes = info.get('nome_citacoes', '')

#         if not id_lattes or not nome_completo:
#             continue

#         producao_bibliografica = dados_aluno.get('producao_bibliografica', {})
#         artigos_periodicos = producao_bibliografica.get('artigos_periodicos', []) or []
#         trabalhos_congressos = producao_bibliografica.get('trabalhos_completos_congressos', []) or []

#         paper_periodicos = extrair_papers_da_lista(
#             artigos_periodicos,
#             tipo_paper='artigos_periodicos',
#             id_lattes_aluno=str(id_lattes),
#             nome_completo_aluno=str(nome_completo).strip(),
#         )
#         paper_congressos = extrair_papers_da_lista(
#             trabalhos_congressos,
#             tipo_paper='trabalhos_completos_congressos',
#             id_lattes_aluno=str(id_lattes),
#             nome_completo_aluno=str(nome_completo).strip(),
#         )

#         citacoes = [item.strip() for item in str(nome_citacoes).split(';') if item.strip()]
#         variacoes_normalizadas = [normalizar_texto(nome_completo)]
#         variacoes_normalizadas.extend(
#             variacao for variacao in (normalizar_texto(citacao) for citacao in citacoes)
#             if variacao and variacao not in variacoes_normalizadas
#         )

#         registros_alunos.append({
#             'id_lattes': str(id_lattes),
#             'nome_completo': str(nome_completo).strip(),
#             'nome_citacoes': str(nome_citacoes).strip(),
#             'nome_completo_normalizado': normalizar_texto(nome_completo),
#             'nome_citacoes_normalizadas': ' | '.join(variacoes_normalizadas[1:]),
#             'variacoes_coautoria': ' | '.join(variacoes_normalizadas),
#             'qtd_artigos_periodicos': len(paper_periodicos),
#             'qtd_trabalhos_congressos': len(paper_congressos),
#             'qtd_papers_extraidos': len(paper_periodicos) + len(paper_congressos),
#         })

#         registros_papers.extend(paper_periodicos)
#         registros_papers.extend(paper_congressos)

#     df_alunos_local = pd.DataFrame(registros_alunos)
#     if not df_alunos_local.empty:
#         df_alunos_local = df_alunos_local.drop_duplicates(subset=['id_lattes']).copy()

#     df_papers_alunos_local = pd.DataFrame(registros_papers)
#     if not df_papers_alunos_local.empty:
#         df_papers_alunos_local = df_papers_alunos_local.drop_duplicates(
#             subset=['id_lattes_aluno', 'tipo_paper', 'titulo_key', 'doi_key']
#         ).copy()

#     return df_alunos_local, df_papers_alunos_local


# # Execução da etapa 1
# df_alunos, df_papers_alunos = extrair_alunos_e_papers('dados_brutos/alunos')
# print(f"Total de alunos carregados: {len(df_alunos)}")
# print(f"Total de papers extraídos dos alunos: {len(df_papers_alunos)}")

# display(df_alunos.head(10))
# display(df_papers_alunos.head(10))

In [41]:
# import numpy as np
# import pandas as pd

# print("ETAPA 2: Preparando chaves globais e referências dos papers dos alunos...")


# def consolidar_referencias_papers(df_papers, tipo_paper):
#     """Consolida os papers extraídos dos alunos em uma tabela de referência única."""
#     df_tipo = df_papers[df_papers['tipo_paper'] == tipo_paper].copy()

#     if df_tipo.empty:
#         return df_tipo

#     def lista_unica(serie):
#         valores = [valor for valor in serie if pd.notna(valor) and str(valor).strip()]
#         return sorted(set(valores))

#     df_tipo = (
#         df_tipo.groupby(['doi_key', 'titulo_key'], as_index=False)
#         .agg(
#             id_lattes_alunos=('id_lattes_aluno', lista_unica),
#             nomes_alunos=('nome_completo_aluno', lista_unica),
#             titulos_originais=('titulo_paper_original', lista_unica),
#             dois_originais=('doi_paper_original', lista_unica),
#             anos=('ano_paper', lista_unica),
#             veiculos=('veiculo_paper', lista_unica),
#             qtd_referencias=('id_lattes_aluno', 'size'),
#         )
#     )

#     return df_tipo


# # Garante que as bases globais tenham as chaves de match prontas e a coluna de coautoria inicializada
# for df_global, titulo_col, doi_col in [
#     (df_artigos_final, 'titulo_artigo', 'doi'),
#     (df_artigos_congresso_final, 'titulo_artigo', 'doi'),
# ]:
#     if 'coautoria_aluno' not in df_global.columns:
#         df_global['coautoria_aluno'] = False
#     else:
#         df_global['coautoria_aluno'] = df_global['coautoria_aluno'].fillna(False).astype(bool)

#     if titulo_col in df_global.columns:
#         df_global['titulo_key_coautoria'] = df_global[titulo_col].apply(normalizar_texto)
#     else:
#         df_global['titulo_key_coautoria'] = ''

#     if doi_col in df_global.columns:
#         df_global['doi_key_coautoria'] = df_global[doi_col].apply(normalizar_doi)
#     else:
#         df_global['doi_key_coautoria'] = ''


# # Consolida as referências por tipo de paper
# if not df_papers_alunos.empty:
#     df_refs_periodicos = consolidar_referencias_papers(df_papers_alunos, 'artigos_periodicos')
#     df_refs_congressos = consolidar_referencias_papers(df_papers_alunos, 'trabalhos_completos_congressos')
# else:
#     df_refs_periodicos = pd.DataFrame()
#     df_refs_congressos = pd.DataFrame()

# print(f"Referências consolidadas - periódicos: {len(df_refs_periodicos)}")
# print(f"Referências consolidadas - congressos: {len(df_refs_congressos)}")

# # Auditoria leve das chaves normalizadas
# if not df_refs_periodicos.empty:
#     display(df_refs_periodicos.head(5))
# if not df_refs_congressos.empty:
#     display(df_refs_congressos.head(5))

In [42]:
# import numpy as np
# import pandas as pd

# print("ETAPA 3: Aplicando o match objetivo por título ou DOI...")


# def marcar_coautoria_por_referencia(df_global, df_refs, *, tipo_base):
#     """Marca coautoria apenas nos registros ainda não sinalizados."""
#     if df_global.empty:
#         return df_global.copy(), pd.DataFrame()

#     df_saida = df_global.copy()
#     df_saida['coautoria_aluno'] = df_saida.get('coautoria_aluno', False)
#     df_saida['coautoria_aluno'] = df_saida['coautoria_aluno'].fillna(False).astype(bool)

#     if df_refs.empty:
#         return df_saida, pd.DataFrame()

#     refs_validas = df_refs.copy()
#     refs_validas['titulo_key'] = refs_validas['titulo_key'].fillna('').astype(str)
#     refs_validas['doi_key'] = refs_validas['doi_key'].fillna('').astype(str)

#     titulos_referencia = set(refs_validas.loc[refs_validas['titulo_key'] != '', 'titulo_key'])
#     dois_referencia = set(refs_validas.loc[refs_validas['doi_key'] != '', 'doi_key'])

#     if 'titulo_key_coautoria' not in df_saida.columns:
#         df_saida['titulo_key_coautoria'] = df_saida['titulo_artigo'].apply(normalizar_texto)
#     if 'doi_key_coautoria' not in df_saida.columns and 'doi' in df_saida.columns:
#         df_saida['doi_key_coautoria'] = df_saida['doi'].apply(normalizar_doi)
#     elif 'doi_key_coautoria' not in df_saida.columns:
#         df_saida['doi_key_coautoria'] = ''

#     candidatos = df_saida.loc[~df_saida['coautoria_aluno']].copy()
#     if candidatos.empty:
#         df_saida = df_saida.drop(columns=['titulo_key_coautoria', 'doi_key_coautoria'], errors='ignore')
#         return df_saida, pd.DataFrame()

#     if titulos_referencia:
#         mascara_titulo = candidatos['titulo_key_coautoria'].isin(titulos_referencia)
#     else:
#         mascara_titulo = pd.Series(False, index=candidatos.index)

#     if dois_referencia:
#         mascara_doi = candidatos['doi_key_coautoria'].isin(dois_referencia)
#     else:
#         mascara_doi = pd.Series(False, index=candidatos.index)

#     mascara_match = mascara_doi | mascara_titulo
#     df_saida.loc[candidatos.index, 'coautoria_aluno'] = mascara_match.to_numpy() | df_saida.loc[candidatos.index, 'coautoria_aluno'].to_numpy()

#     lookup_doi = refs_validas.loc[refs_validas['doi_key'] != ''].groupby('doi_key')['id_lattes_alunos'].first().to_dict()
#     lookup_titulo = refs_validas.loc[refs_validas['titulo_key'] != ''].groupby('titulo_key')['id_lattes_alunos'].first().to_dict()

#     auditoria = candidatos.loc[mascara_match, [c for c in ['id_lattes', 'titulo_artigo', 'doi', 'coautoria_aluno', 'titulo_key_coautoria', 'doi_key_coautoria'] if c in candidatos.columns]].copy()
#     auditoria['tipo_base'] = tipo_base
#     auditoria['motivo_match'] = np.where(mascara_doi.loc[mascara_match], 'doi', 'titulo')
#     auditoria['alunos_referencia'] = auditoria.apply(
#         lambda linha: lookup_doi.get(linha.get('doi_key_coautoria', ''), lookup_titulo.get(linha.get('titulo_key_coautoria', ''), [])),
#         axis=1,
#     )

#     df_saida = df_saida.drop(columns=['titulo_key_coautoria', 'doi_key_coautoria'], errors='ignore')
#     return df_saida, auditoria


# # Periódicos
# print("Marcando coautoria em periódicos...")
# df_artigos_final, df_auditoria_periodicos = marcar_coautoria_por_referencia(
#     df_artigos_final,
#     df_refs_periodicos,
#     tipo_base='periodico',
# )

# # Conferências
# print("Marcando coautoria em conferências...")
# df_artigos_congresso_final, df_auditoria_congressos = marcar_coautoria_por_referencia(
#     df_artigos_congresso_final,
#     df_refs_congressos,
#     tipo_base='conferencia',
# )

# # Base de auditoria consolidada
# df_auditoria_coautoria = pd.concat(
#     [df_auditoria_periodicos, df_auditoria_congressos],
#     ignore_index=True,
# ) if not df_auditoria_periodicos.empty or not df_auditoria_congressos.empty else pd.DataFrame()

# print("\n--- RESULTADO FINAL ---")
# print(f"Periódicos com coautoria de aluno: {int(df_artigos_final['coautoria_aluno'].sum())}")
# print(f"Conferências com coautoria de aluno: {int(df_artigos_congresso_final['coautoria_aluno'].sum())}")
# print(f"Registros auditados nesta rodada: {len(df_auditoria_coautoria)}")

# if not df_auditoria_coautoria.empty:
#     display(df_auditoria_coautoria.head(10))
# else:
#     print("Nenhum novo match foi encontrado nesta rodada.")

In [43]:
# print("ETAPA 4: Auditoria final das marcas de coautoria...")

# resumo_periodicos = pd.DataFrame({
#     'total_registros': [len(df_artigos_final)],
#     'coautoria_aluno_true': [int(df_artigos_final['coautoria_aluno'].sum())],
#     'coautoria_aluno_false': [int((~df_artigos_final['coautoria_aluno']).sum())],
# })

# resumo_congressos = pd.DataFrame({
#     'total_registros': [len(df_artigos_congresso_final)],
#     'coautoria_aluno_true': [int(df_artigos_congresso_final['coautoria_aluno'].sum())],
#     'coautoria_aluno_false': [int((~df_artigos_congresso_final['coautoria_aluno']).sum())],
# })

# print("Resumo periódicos:")
# display(resumo_periodicos)
# print("Resumo conferências:")
# display(resumo_congressos)

# if not df_auditoria_coautoria.empty:
#     print("Amostra da auditoria consolidada:")
#     display(df_auditoria_coautoria[[c for c in ['tipo_base', 'motivo_match', 'titulo_artigo', 'doi', 'alunos_referencia'] if c in df_auditoria_coautoria.columns]].head(10))

# print("As bases globais ficaram prontas para a persistência no DuckDB.")

In [44]:
# print("ETAPA 5: Fallback complementar por autores para maior cobertura...")


# def preparar_variacoes_autores(df_alunos_local):
#     """Monta a lista de variações de nome dos alunos para busca por string de autores."""
#     variacoes = []

#     if df_alunos_local.empty:
#         return variacoes

#     for _, linha in df_alunos_local.iterrows():
#         nomes = [item.strip() for item in str(linha.get('variacoes_coautoria', '')).split(' | ') if item.strip()]
#         if nomes:
#             variacoes.append(set(nomes))

#     return variacoes


# def tem_coautoria_aluno_por_autores(autores_paper_str, variacoes_alunos_lista):
#     """Verifica se a lista de autores contém algum aluno."""
#     if pd.isna(autores_paper_str) or not str(autores_paper_str).strip() or not variacoes_alunos_lista:
#         return False

#     autores_normalizados = f" {normalizar_texto(autores_paper_str)} "

#     for conjunto_variacoes_aluno in variacoes_alunos_lista:
#         for nome_aluno in conjunto_variacoes_aluno:
#             if f" {nome_aluno} " in autores_normalizados:
#                 return True

#     return False


# variacoes_alunos_autores = preparar_variacoes_autores(df_alunos)
# auditoria_autores = []

# # Periódicos: tenta recuperar a coluna 'autores' se ela estiver disponível em uma fonte anterior do notebook.
# if 'autores' not in df_artigos_final.columns and 'df_sucessos' in globals() and {'titulo', 'autores', 'id_lattes'}.issubset(df_sucessos.columns):
#     df_artigos_final = df_artigos_final.merge(
#         df_sucessos[['id_lattes', 'titulo', 'autores']].drop_duplicates(),
#         left_on=['id_lattes', 'titulo_artigo'] if 'titulo_artigo' in df_artigos_final.columns else ['id_lattes', 'titulo'],
#         right_on=['id_lattes', 'titulo'],
#         how='left'
#     )
#     if 'titulo' in df_artigos_final.columns:
#         df_artigos_final = df_artigos_final.drop(columns=['titulo'])

# if 'autores' in df_artigos_final.columns:
#     mask_periodicos = ~df_artigos_final['coautoria_aluno'].fillna(False).astype(bool)
#     matches_periodicos = df_artigos_final.loc[mask_periodicos, 'autores'].apply(
#         lambda autores: tem_coautoria_aluno_por_autores(autores, variacoes_alunos_autores)
#     )
#     if matches_periodicos.any():
#         df_artigos_final.loc[matches_periodicos.index[matches_periodicos], 'coautoria_aluno'] = True
#         auditoria_periodicos_autores = df_artigos_final.loc[
#             matches_periodicos.index[matches_periodicos],
#             [col for col in ['id_lattes', 'titulo_artigo', 'doi', 'autores'] if col in df_artigos_final.columns]
#         ].copy()
#         auditoria_periodicos_autores['tipo_base'] = 'periodico'
#         auditoria_periodicos_autores['motivo_match'] = 'autores'
#         auditoria_autores.append(auditoria_periodicos_autores)

# # Conferências: usa diretamente a coluna 'autores' já existente.
# if 'autores' in df_artigos_congresso_final.columns:
#     mask_congressos = ~df_artigos_congresso_final['coautoria_aluno'].fillna(False).astype(bool)
#     matches_congressos = df_artigos_congresso_final.loc[mask_congressos, 'autores'].apply(
#         lambda autores: tem_coautoria_aluno_por_autores(autores, variacoes_alunos_autores)
#     )
#     if matches_congressos.any():
#         df_artigos_congresso_final.loc[matches_congressos.index[matches_congressos], 'coautoria_aluno'] = True
#         auditoria_congressos_autores = df_artigos_congresso_final.loc[
#             matches_congressos.index[matches_congressos],
#             [col for col in ['id_lattes', 'titulo_artigo', 'doi', 'autores'] if col in df_artigos_congresso_final.columns]
#         ].copy()
#         auditoria_congressos_autores['tipo_base'] = 'conferencia'
#         auditoria_congressos_autores['motivo_match'] = 'autores'
#         auditoria_autores.append(auditoria_congressos_autores)

# if auditoria_autores:
#     df_auditoria_autores = pd.concat(auditoria_autores, ignore_index=True)
# else:
#     df_auditoria_autores = pd.DataFrame()

# print("\n--- COMPLEMENTO POR AUTORES ---")
# print(f"Periódicos com coautoria de aluno: {int(df_artigos_final['coautoria_aluno'].sum())}")
# print(f"Conferências com coautoria de aluno: {int(df_artigos_congresso_final['coautoria_aluno'].sum())}")
# print(f"Registros adicionais detectados por autores: {len(df_auditoria_autores)}")

# if not df_auditoria_autores.empty:
#     display(df_auditoria_autores.head(10))

# Conectando com DuckDB

In [45]:
df_artigos_final.head()

,id_lattes,titulo_artigo,titulo_revista_lattes,ano_pub,doi,autores,match_adequado,id_scopus,titulo_revista_scopus,maior_percentil,codigo_area_maior_percentil,area_maior_percentil,issn,computation_area,coautoria_aluno
0,0211300683784278,On the (In)Dependence of the Peano Axioms for ...,HISTORY AND PHILOSOPHY OF LOGIC,2021,http://dx.doi.org/10.1080/01445340.2021.1971005,"CERIOLI, MÁRCIA R.; NOBREGA, HUGO ; SILVEIRA, ...",True,4700152837,HISTORY AND PHILOSOPHY OF LOGIC,82,1202,History,14645149,False,False
1,0211300683784278,Short proofs on the structure of general parti...,DISCRETE APPLIED MATHEMATICS,2021,http://dx.doi.org/10.1016/j.dam.2020.09.007,"CERIOLI, MÁRCIA R.; MARTINS, TAÍSA",True,25890,DISCRETE APPLIED MATHEMATICS,73,2607,Discrete Mathematics and Combinatorics,NaN,False,False
2,0211300683784278,Transversals of longest paths,DISCRETE MATHEMATICS,2020,http://dx.doi.org/10.1016/j.disc.2019.111717,"CERIOLI, MÁRCIA R.; FERNANDES, CRISTINA G. ; G...",True,25892,DISCRETE MATHEMATICS,57,2607,Discrete Mathematics and Combinatorics,NaN,True,True
3,0211300683784278,Intersection of longest paths in graph classes,DISCRETE APPLIED MATHEMATICS,2020,http://dx.doi.org/10.1016/j.dam.2019.03.022,"CERIOLI, MÁRCIA R.; LIMA, PALOMA T.",True,25890,DISCRETE APPLIED MATHEMATICS,73,2607,Discrete Mathematics and Combinatorics,NaN,False,True
4,0211300683784278,"L(2,1)-labelling of graphs with few P4?s",DISCRETE OPTIMIZATION,2016,NaN,"CERIOLI, M. R.; POSNER, D. F. D. ; Martins, N....",True,28469,DISCRETE OPTIMIZATION,58,2604,Applied Mathematics,NaN,True,True


In [46]:
import duckdb

print("Iniciando persistência no DuckDB com as tabelas de Artigos, Alunos e Orientações...")
con = duckdb.connect('pesquisadores_oficial.duckdb')

# ==========================================
# 1. CRIAÇÃO DOS SCHEMAS E SEQUÊNCIAS
# ==========================================

# Tabela Mãe: Professores
query_cria_pessoas = """
CREATE TABLE IF NOT EXISTS tb_professores (
    id_lattes VARCHAR PRIMARY KEY,
    nome_completo VARCHAR,
    nome_citacoes VARCHAR,
    sexo VARCHAR,
    rotulo VARCHAR,
    periodo VARCHAR,
    bolsa_produtividade VARCHAR,
    endereco_profissional VARCHAR,
    atualizacao_cv TIMESTAMP,
    url VARCHAR,
    texto_resumo VARCHAR
);
"""
con.execute(query_cria_pessoas)

# Tabela nova: Alunos
query_cria_alunos = """
CREATE TABLE IF NOT EXISTS tb_alunos (
    id_lattes VARCHAR PRIMARY KEY,
    nome_completo VARCHAR,
    nome_citacoes VARCHAR,
    nome_completo_normalizado VARCHAR,
    nome_citacoes_normalizadas VARCHAR,
    variacoes_coautoria VARCHAR
);
"""
con.execute(query_cria_alunos)

# Sequências para os IDs automáticos das três tabelas filhas
con.execute("CREATE SEQUENCE IF NOT EXISTS seq_id_artigo_periodico;")
con.execute("CREATE SEQUENCE IF NOT EXISTS seq_id_artigo_conferencia;")
con.execute("CREATE SEQUENCE IF NOT EXISTS seq_id_orientacao;")

# Tabela Filha 1: Artigos de Periódicos (Scopus)
query_cria_periodicos = """
CREATE TABLE IF NOT EXISTS tb_artigo_periodico (
    id_artigo_periodico INTEGER PRIMARY KEY DEFAULT nextval('seq_id_artigo_periodico'),
    id_lattes VARCHAR,
    titulo_artigo VARCHAR NOT NULL,
    titulo_revista_lattes VARCHAR,
    ano_pub INTEGER,
    doi VARCHAR,
    autores VARCHAR,
    match_adequado BOOLEAN,
    coautoria_aluno BOOLEAN,
    id_scopus VARCHAR,
    titulo_revista_scopus VARCHAR,
    maior_percentil INTEGER,
    codigo_area_maior_percentil VARCHAR,
    area_maior_percentil VARCHAR,
    issn VARCHAR,
    computation_area BOOLEAN,
    FOREIGN KEY (id_lattes) REFERENCES tb_professores(id_lattes)
);
"""
con.execute(query_cria_periodicos)

# Tabela Filha 2: Artigos de Conferências/Congressos (Google)
query_cria_conferencias = """
CREATE TABLE IF NOT EXISTS tb_artigo_conferencia (
    id_artigo_conferencia INTEGER PRIMARY KEY DEFAULT nextval('seq_id_artigo_conferencia'),
    id_lattes VARCHAR,
    titulo_artigo VARCHAR NOT NULL,
    ano INTEGER,
    doi VARCHAR,
    autores VARCHAR,
    titulo_evento_lattes VARCHAR,
    paginas VARCHAR,
    sigla_evento_google VARCHAR,
    titulo_evento_google VARCHAR,
    estrato VARCHAR,
    tipo_match VARCHAR,
    coautoria_aluno BOOLEAN,
    FOREIGN KEY (id_lattes) REFERENCES tb_professores(id_lattes)
);
"""
con.execute(query_cria_conferencias)

# Tabela Filha 3: Orientações
query_cria_orientacoes = """
CREATE TABLE IF NOT EXISTS tb_orientacoes (
    id_orientacao INTEGER PRIMARY KEY DEFAULT nextval('seq_id_orientacao'),
    id_lattes VARCHAR,
    titulo_trabalho VARCHAR,
    ano_inicio INTEGER,
    orientando VARCHAR,
    tipo_trabalho VARCHAR,
    instituicao VARCHAR,
    curso VARCHAR,
    status VARCHAR,
    nivel VARCHAR,
    ano_conclusao INTEGER,
    FOREIGN KEY (id_lattes) REFERENCES tb_professores(id_lattes)
);
"""
con.execute(query_cria_orientacoes)

# Garante que tabelas antigas recebam as colunas novas quando o notebook for reexecutado
for alter_sql in [
    "ALTER TABLE tb_artigo_periodico ADD COLUMN autores VARCHAR",
    "ALTER TABLE tb_artigo_periodico ADD COLUMN doi VARCHAR",
    "ALTER TABLE tb_artigo_periodico ADD COLUMN coautoria_aluno BOOLEAN",
    "ALTER TABLE tb_artigo_conferencia ADD COLUMN coautoria_aluno BOOLEAN",
    "ALTER TABLE tb_alunos ADD COLUMN nome_completo_normalizado VARCHAR",
    "ALTER TABLE tb_alunos ADD COLUMN nome_citacoes_normalizadas VARCHAR",
    "ALTER TABLE tb_alunos ADD COLUMN variacoes_coautoria VARCHAR",
]:
    try:
        con.execute(alter_sql)
    except Exception:
        pass

print("Tabelas criadas com sucesso (ou já existentes).")

# ==========================================
# 2. INSERÇÃO DOS DADOS (Carga via Pandas)
# ==========================================
print("Limpando dados antigos (Filhas primeiro, mãe depois)...")
# Apagar as filhas antes da mãe para não violar a integridade relacional
con.execute("DELETE FROM tb_artigo_periodico")
con.execute("DELETE FROM tb_artigo_conferencia")
con.execute("DELETE FROM tb_orientacoes")
con.execute("DELETE FROM tb_alunos")
con.execute("DELETE FROM tb_professores")

print("Inserindo novos dados a partir dos DataFrames Pandas...")

# Inserção da Tabela Mãe (Professores)
if not df_pessoas.empty:
    con.execute("INSERT INTO tb_professores SELECT * FROM df_pessoas")

# Inserção da Tabela de Alunos
if not df_alunos.empty:
    con.execute("""
        INSERT INTO tb_alunos (
            id_lattes, nome_completo, nome_citacoes,
            nome_completo_normalizado, nome_citacoes_normalizadas, variacoes_coautoria
        )
        SELECT
            id_lattes, nome_completo, nome_citacoes,
            nome_completo_normalizado, nome_citacoes_normalizadas, variacoes_coautoria
        FROM df_alunos
    """)

# Inserção da Tabela Filha 1 (Artigos Periódicos)
if not df_artigos_final.empty:
    con.execute("""
        INSERT INTO tb_artigo_periodico (
            id_lattes, titulo_artigo, titulo_revista_lattes, ano_pub,
            doi, autores, match_adequado, coautoria_aluno, id_scopus,
            titulo_revista_scopus, maior_percentil, codigo_area_maior_percentil,
            area_maior_percentil, issn, computation_area
        )
        SELECT
            id_lattes, titulo_artigo, titulo_revista_lattes, ano_pub,
            doi, autores, match_adequado, coautoria_aluno, id_scopus,
            titulo_revista_scopus, maior_percentil, codigo_area_maior_percentil,
            area_maior_percentil, issn, computation_area
        FROM df_artigos_final
    """)

# Inserção da Tabela Filha 2 (Artigos Conferência)
if not df_artigos_congresso_final.empty:
    con.execute("""
        INSERT INTO tb_artigo_conferencia (
            id_lattes, titulo_artigo, ano, doi, autores,
            titulo_evento_lattes, paginas, sigla_evento_google,
            titulo_evento_google, estrato, tipo_match, coautoria_aluno
        )
        SELECT
            id_lattes, titulo_artigo, ano, doi, autores,
            titulo_evento_lattes, paginas, sigla_evento_google,
            titulo_evento_google, estrato, tipo_match, coautoria_aluno
        FROM df_artigos_congresso_final
    """)

# Inserção da Tabela Filha 3 (Orientações)
if not df_orientacoes.empty:
    con.execute("""
        INSERT INTO tb_orientacoes (
            id_lattes, titulo_trabalho, ano_inicio, orientando,
            tipo_trabalho, instituicao, curso, status, nivel, ano_conclusao
        )
        SELECT
            id_lattes, titulo_trabalho, ano_inicio, orientando,
            tipo_trabalho, instituicao, curso, status, nivel, ano_conclusao
        FROM df_orientacoes
    """)

con.close()
print("Processo finalizado! Banco 'pesquisadores.duckdb' atualizado com o schema completo.")

Iniciando persistência no DuckDB com as tabelas de Artigos, Alunos e Orientações...
Tabelas criadas com sucesso (ou já existentes).
Limpando dados antigos (Filhas primeiro, mãe depois)...
Inserindo novos dados a partir dos DataFrames Pandas...
Processo finalizado! Banco 'pesquisadores.duckdb' atualizado com o schema completo.
